# Load libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# !pip install pandas numpy matplotlib seaborn openpyxl statsmodels plotly
# !pip install keras-tuner
# !pip install pmdarima
# !pip install --upgrade numpy
# !pip install --upgrade tensorflow
# !pip install scikit-optimize
# !pip install statsmodels scipy

  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.25.0
    Uninstalling numpy-1.25.0:
      Successfully uninstalled numpy-1.25.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.19.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.19.0 which is incompatible.
  Using cached numpy-2.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x8

# Load data

In [ ]:
# Load the Accessories sheet from the Excel file
outdoor_data = pd.read_excel('Outdoor.xlsx')

# Display the first few rows to inspect the data
outdoor_data.head()

,Customer Name,Reporting Business Type,Collective Class,Series No,Item SKU,Item Description,Colors,Series Color,Item Grouping,AFI Sales Category,Import/Domestic Code,Market Introduced At,Parent Style Description,Warehouse,Order Quantity,FiscalMonthLastDate,Status
0,Customer 1,US,OUTDOOR,P312-049,P312-049,CHAIRS W/TABLE SET (3/CN),Gray,Multi,Outdoor Dining,Sig. Motion Outdoor,I,QTR 4 2021,Casual,1,2.0,2023-04-29,Invoiced
1,Customer 3,US,OUTDOOR,P459,P459-877,CORNER WITH CUSHION (2/CN),Gray,Gray,Outdoor Seating,Sig. Motion Outdoor,I,QTR 2 2021,Casual,17,2.0,2023-05-27,Invoiced
2,Customer 4,e-Commerce,OUTDOOR,P730,P730-615,ROUND DINING TABLE W/UMB OPT,Brown,Brown,Outdoor Dining,Sig. Motion Outdoor,I,QTR 2 2021,Casual,1,3.0,2024-06-29,Invoiced
3,Customer 5,US,OUTDOOR,P220,P220-115,DINING TABLE SET (3/CN),Brown/Black,Brown/Black,Outdoor Dining,Sig. Motion Outdoor,I,Las Vegas July 2020,Casual,5,9.0,2022-03-26,Invoiced
4,Customer 6,US,OUTDOOR,P334,P334-081,SOFA/CHAIRS/TABLE SET (4/CN),Gray,Gray,Outdoor Seating,Sig. Motion Outdoor,I,Las Vegas July 2019,Contemporary,1,2.0,2022-07-23,Invoiced


# Convert data from long format to wide format

In [ ]:
import pandas as pd

# -------------------------------
# Step 0: Data Preparation and Cleaning
# -------------------------------
df = outdoor_data[[
    "FiscalMonthLastDate", "Warehouse", "Series No", "Item SKU", "Order Quantity"
]].copy().dropna()

# Đổi tên cột: thay khoảng trắng bằng dấu "-" để dễ xử lý
df.columns = [col.replace(" ", "-") for col in df.columns]

# Chuyển FiscalMonthLastDate sang kiểu datetime
df["FiscalMonthLastDate"] = pd.to_datetime(df["FiscalMonthLastDate"])

# -------------------------------
# Step 1: Aggregate Data at the Series Level
# -------------------------------
# Chúng ta chỉ quan tâm đến cột "Series No". Mỗi hàng sẽ được gộp theo FiscalMonthLastDate và Series-No.
df_agg = df.groupby(["FiscalMonthLastDate", "Series-No"])["Order-Quantity"].sum().reset_index()

# -------------------------------
# Step 2: Convert to Wide Format for Forecasting
# -------------------------------
# Pivot table: index = FiscalMonthLastDate, các cột là các giá trị của "Series-No"
df_wide = df_agg.pivot_table(
    index="FiscalMonthLastDate",
    columns="Series-No",
    values="Order-Quantity",
    fill_value=0
)

# -------------------------------
# Step 3: Compute Hierarchical Aggregations (2-level: Total and Series)
# -------------------------------
# 3.1: Total (Overall Sum): tổng của tất cả các Series
df_total = df_wide.sum(axis=1).to_frame(name=("Total",))

# 3.2: Series-level: giữ lại dữ liệu của các Series, đặt tên cột thành MultiIndex (1 cấp)
df_series = df_wide.copy()
df_series.columns = pd.MultiIndex.from_tuples([(col,) for col in df_series.columns])

# -------------------------------
# Step 4: Merge Total and Series-level Data
# -------------------------------
df_hierarchical = pd.concat([df_total, df_series], axis=1)

# -------------------------------
# Step 5: Sort Columns Hierarchically
# -------------------------------
# Sắp xếp sao cho cột "Total" đứng đầu, sau đó các Series được sắp xếp theo tên (alphabet)
ordered_columns = sorted(
    df_hierarchical.columns,
    key=lambda x: (0 if x[0] == "Total" else 1, x[0])
)
df_hierarchical = df_hierarchical[ordered_columns]

# -------------------------------
# Step 6: Flatten MultiIndex Columns
# -------------------------------
# Vì các cột Series đã có MultiIndex 1 cấp, ta chỉ lấy phần tử đầu tiên của mỗi tuple
df_hierarchical.columns = [col[0] for col in df_hierarchical.columns]

# -------------------------------
# (Optional) Replace specific dates in the index if needed
# -------------------------------
df_hierarchical.index = df_hierarchical.index.to_series().replace({
    pd.Timestamp("2023-04-01"): pd.Timestamp("2023-03-25"),
    pd.Timestamp("2023-07-01"): pd.Timestamp("2023-06-24")
})

# Categorize series into X,Y,Z levels

In [ ]:
# -------------------------------
# Phần 2: Phân loại các Series thành nhóm a, b, c theo doanh số
# -------------------------------
series_cols = [col for col in df_hierarchical.columns if col != "Total"]

# Tính tổng doanh số cho mỗi Series
series_totals = df_hierarchical[series_cols].sum(axis=0).reset_index()
series_totals.columns = ['Series-No', 'Total_Sales']
overall_total = series_totals['Total_Sales'].sum()
n_total = len(series_totals)

# Sắp xếp giảm dần theo Total_Sales
series_totals = series_totals.sort_values(by='Total_Sales', ascending=False).reset_index(drop=True)

# Nhóm a: chọn các series theo thứ tự giảm dần cho đến khi tích lũy ≥80% overall sales,
#         nhưng số lượng không vượt quá 20% tổng số series.
max_count_a = int(0.2 * n_total)
cumulative_sales_a = 0
group_a = []
for idx, row in series_totals.iterrows():
    group_a.append(row['Series-No'])
    cumulative_sales_a += row['Total_Sales']
    if (cumulative_sales_a / overall_total >= 0.80) or (len(group_a) >= max_count_a):
        break

# Nhóm b: từ các series còn lại, chọn đến khi tích lũy ≥15% overall sales,
#         nhưng số lượng không vượt quá 30% số series còn lại.
remaining_df = series_totals[~series_totals['Series-No'].isin(group_a)].copy()
n_remaining = len(remaining_df)
max_count_b = int(0.3 * n_remaining)
cumulative_sales_b = 0
group_b = []
for idx, row in remaining_df.iterrows():
    group_b.append(row['Series-No'])
    cumulative_sales_b += row['Total_Sales']
    if (cumulative_sales_b / overall_total >= 0.15) or (len(group_b) >= max_count_b):
        break

# Nhóm c: các series còn lại
group_c = remaining_df[~remaining_df['Series-No'].isin(group_b)]['Series-No'].tolist()

series_totals['Group'] = series_totals['Series-No'].apply(
    lambda x: 'a' if x in group_a else ('b' if x in group_b else 'c')
)

print("\nPhân nhóm doanh số (a, b, c):")
print("Tổng số Series:", n_total)
print("Nhóm a: {} Series, chiếm {:.2%} doanh số".format(len(group_a), cumulative_sales_a / overall_total))
print("Nhóm b: {} Series, chiếm {:.2%} doanh số".format(len(group_b), cumulative_sales_b / overall_total))
remaining_sales = overall_total - cumulative_sales_a - cumulative_sales_b
print("Nhóm c: {} Series, chiếm {:.2%} doanh số".format(len(group_c), remaining_sales / overall_total))



# -------------------------------
# Phần 3: Phân nhóm bổ sung x, y, z trong mỗi nhóm a, b, c theo hệ số biến động (CV)
# -------------------------------
# Tính CV từ dữ liệu 12 tháng cuối của wide format
last_12 = df_hierarchical.iloc[-12:]
def compute_cv(series_data):
    mean_val = series_data.mean()
    return np.nan if mean_val == 0 else series_data.std() / mean_val

# Tính CV cho tất cả các series
cv_dict = {series: compute_cv(last_12[series]) for series in series_cols}

# Áp dụng ngưỡng phân loại CV: X: CV < 0.34, Y: 0.34 ≤ CV < 0.84, Z: CV ≥ 0.84.
subgroup_dict = {}
for series, cv_val in cv_dict.items():
    if pd.isna(cv_val):
        subgroup = 'unknown'
    elif cv_val < 0.34:
        subgroup = 'x'
    elif cv_val < 0.84:
        subgroup = 'y'
    else:
        subgroup = 'z'
    subgroup_dict[series] = subgroup

# In bảng tóm tắt số lượng các nhóm phụ
from collections import Counter
subgroup_counts = Counter(subgroup_dict.values())
print("\nPhân loại phụ theo CV:")
for key in ['x', 'y', 'z', 'unknown']:
    print(f"Nhóm {key}: {subgroup_counts.get(key, 0)} series")

# -------------------------------
# Phần 4: Gán nhãn kết hợp vào tên cột wide format
# -------------------------------
# Tạo mapping cuối: tên cột sẽ có định dạng <Series-No>_<Group (a,b,c)>_<Subgroup (x,y,z,unknown)>
final_mapping = {}
for series in series_cols:
    # Lấy nhãn nhóm a,b,c từ series_totals
    group_label = series_totals.loc[series_totals['Series-No'] == series, 'Group'].values[0]
    subgroup_label = subgroup_dict.get(series, '')
    final_mapping[series] = f"{series}_{group_label}_{subgroup_label}"

# Cập nhật tên cột trong df_hierarchical (cột Total giữ nguyên)
new_columns = {}
for col in df_hierarchical.columns:
    if col == "Total":
        new_columns[col] = col
    else:
        new_columns[col] = final_mapping.get(col, col)

df_final = df_hierarchical.rename(columns=new_columns)


# -------------------------------
# Phần 5: Tách Stop Produced Series và Valid Series
# -------------------------------
# Lấy dữ liệu 12 tháng cuối
last_12 = df_final.iloc[-12:]

# Xác định các cột mà toàn bộ 12 giá trị cuối cùng đều bằng 0
stop_columns = [col for col in df_final.columns if (last_12[col] == 0).all()]

# Tạo DataFrame chứa các stop produced series và valid series
df_zero = df_final[stop_columns]
df_valid = df_final.drop(columns=stop_columns)

# In ra số lượng các series từng loại
print("Số lượng stop produced series:", len(stop_columns))
print("Số lượng valid series:", len(df_valid.columns) - (1 if 'Total' in df_valid.columns else 0)-1)

# -------------------------------
# Lưu kết quả thành 2 file CSV
# -------------------------------
df_zero.to_csv("Stopped_Series_Hierarchical.csv", index=True)
df_valid.to_csv("Series_Hierarchical.csv", index=True)


Phân nhóm doanh số (a, b, c):
Tổng số Series: 235
Nhóm a: 47 Series, chiếm 77.65% doanh số
Nhóm b: 40 Series, chiếm 15.10% doanh số
Nhóm c: 148 Series, chiếm 7.25% doanh số

Phân loại phụ theo CV:
Nhóm x: 2 series
Nhóm y: 73 series
Nhóm z: 131 series
Nhóm unknown: 29 series
Số lượng stop produced series: 29
Số lượng valid series: 205


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

# --- Bước 1: Load dữ liệu ---
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# --- Bước 2: Phân loại các series hợp lệ ---
# Loại bỏ cột "Total" và chỉ xét các cột chứa dữ liệu series
valid_series_cols = [col for col in df_final.columns if col != 'Total']

# Giả sử các cột đã được gán nhãn theo định dạng: <Series-No>_<group a,b,c>_<subgroup: x, y, z, unknown>
valid_x = [col for col in valid_series_cols if col.endswith('_x')]
valid_y = [col for col in valid_series_cols if col.endswith('_y')]
valid_z = [col for col in valid_series_cols if col.endswith('_z')]

print("Tổng số series hợp lệ:")
print("Nhóm x:", len(valid_x))
print("Nhóm y:", len(valid_y))
print("Nhóm z:", len(valid_z))

Tổng số series hợp lệ:
Nhóm x: 2
Nhóm y: 73
Nhóm z: 131


# 1. Forecast Model cho Nhóm X (CV < 0.34 – dữ liệu ổn định)

## 1.1 Xây dựng model dự báo cho dòng series X bằng Holt winter & Arima

In [ ]:
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np
import pandas as pd
from scipy.stats import boxcox
from scipy.special import inv_boxcox

# Helper function for computing wMAPE
def compute_wmape(y_true, y_pred):
    return np.sum(np.abs(y_pred - y_true)) / np.sum(y_true)

# Helper function for Box-Cox and Inverse Box-Cox with a shift to ensure positivity
def arima_boxcox_forecast(series, forecast_periods=6):
    # Shift series if it contains zero or negative values
    if (series <= 0).any():
        shift_val = abs(series.min()) + 1  # Ensure all values are positive
        series = series + shift_val
    else:
        shift_val = 0

    # Apply Box-Cox transformation
    series_boxcox, lam = boxcox(series)
    model = ARIMA(series_boxcox, order=(1, 1, 0))  # ARIMA model
    model_fit = model.fit()
    forecast_boxcox = model_fit.forecast(forecast_periods)
    forecast_arima = inv_boxcox(forecast_boxcox, lam)

    # If a shift was applied, subtract it from the forecasted values
    if shift_val > 0:
        forecast_arima -= shift_val

    return forecast_arima

# Helper function for Holt-Winters Exponential Smoothing
def holt_winters_forecast(series, forecast_periods=6, seasonal_periods=12):
    model = ExponentialSmoothing(
        series, trend='add', seasonal='add', seasonal_periods=seasonal_periods
    ).fit()
    forecast = model.forecast(forecast_periods)
    return forecast

# ARIMA Hyperparameter tuning using auto_arima from pmdarima
def arima_tuning(series):
    model = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=True)
    return model

# Ensemble pipeline using Holt-Winters and ARIMA
def ensemble_pipeline_holt_arima(series, forecast_periods=6, seasonal_periods=12):
    """
    This function runs the ensemble pipeline using Holt-Winters and ARIMA models.
    It then optimizes the ensemble weight to minimize test wMAPE (computed on the original scale).

    Returns: (best wMAPE, optimal weight, ARIMA forecast, Holt-Winters forecast)
    """
    # No preprocessing is done here. Directly use the original series.
    if len(series) < 36:
        return None, None, None, None

    # Split data: last forecast_periods for test
    train_orig = series.iloc[:-forecast_periods]
    test_orig  = series.iloc[-forecast_periods:]

    # ---- Model 1: ARIMA with Box-Cox
    best_arima_model = arima_tuning(train_orig)
    forecast_arima = best_arima_model.predict(n_periods=forecast_periods)

    # ---- Model 2: Holt-Winters Exponential Smoothing
    forecast_hw = holt_winters_forecast(train_orig, forecast_periods=forecast_periods, seasonal_periods=seasonal_periods)

    # ---- Evaluate Ensemble
    # We'll use test_orig as the ground truth for both models
    y_true = test_orig.values

    # Optimize weight for ensemble (ARIMA + Holt-Winters)
    weights = np.linspace(0, 1, 101)
    best_wmape = np.inf
    best_weight = None
    for w in weights:
        ensemble_pred = w * forecast_arima + (1 - w) * forecast_hw
        wmape_val = compute_wmape(y_true, ensemble_pred)
        if wmape_val < best_wmape:
            best_wmape = wmape_val
            best_weight = w

    return best_wmape, best_weight, forecast_arima, forecast_hw

# Main block for running the pipeline on each series
df = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)
group_x_cols = [col for col in df.columns if col != 'Total' and col.endswith('_x')]

results = []
for col in group_x_cols:
    s = df[col]
    wmape_val, best_weight, forecast_arima, forecast_hw = ensemble_pipeline_holt_arima(s, forecast_periods=6, seasonal_periods=12)
    results.append({"Series": col, "Ensemble Test wMAPE": wmape_val, "Optimal Weight": best_weight})

results_df = pd.DataFrame(results)
# Format the wMAPE as percentage with two decimals (if not None)
results_df["Ensemble Test wMAPE"] = results_df["Ensemble Test wMAPE"].apply(lambda x: "{:.2%}".format(x) if x is not None else None)

# Display the results using pandas
import pandas as pd
print(results_df)

# Count how many series have ensemble wMAPE below 25%
def parse_percent(pct_str):
    return float(pct_str.strip('%')) / 100 if pct_str is not None else None

results_df["wMAPE_float"] = results_df["Ensemble Test wMAPE"].apply(parse_percent)
valid_count = results_df['wMAPE_float'].dropna().apply(lambda x: x < 0.25).sum()
total_series = results_df['wMAPE_float'].dropna().shape[0]
print(f"\nOut of {total_series} series in Group X, {valid_count} have Ensemble Test wMAPE below 25%.")

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[12] intercept   : AIC=inf, Time=0.44 sec
 ARIMA(0,1,0)(0,0,0)[12] intercept   : AIC=400.353, Time=0.01 sec
 ARIMA(1,1,0)(1,0,0)[12] intercept   : AIC=394.596, Time=0.06 sec
 ARIMA(0,1,1)(0,0,1)[12] intercept   : AIC=387.390, Time=0.18 sec
 ARIMA(0,1,0)(0,0,0)[12]             : AIC=398.615, Time=0.01 sec
 ARIMA(0,1,1)(0,0,0)[12] intercept   : AIC=386.588, Time=0.06 sec
 ARIMA(0,1,1)(1,0,0)[12] intercept   : AIC=inf, Time=0.17 sec
 ARIMA(0,1,1)(1,0,1)[12] intercept   : AIC=inf, Time=0.24 sec
 ARIMA(1,1,1)(0,0,0)[12] intercept   : AIC=388.370, Time=0.08 sec
 ARIMA(0,1,2)(0,0,0)[12] intercept   : AIC=388.370, Time=0.07 sec
 ARIMA(1,1,0)(0,0,0)[12] intercept   : AIC=392.603, Time=0.03 sec
 ARIMA(1,1,2)(0,0,0)[12] intercept   : AIC=inf, Time=0.09 sec
 ARIMA(0,1,1)(0,0,0)[12]             : AIC=388.828, Time=0.02 sec

Best model:  ARIMA(0,1,1)(0,0,0)[12] intercept
Total fit time: 1.478 seconds
Performing stepwise search to minimiz

## 1.2. Dự báo cho series P801 sử dụng Holt winter & Arima

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA


# Helper function for computing wMAPE
def compute_wmape(y_true, y_pred):
    return np.sum(np.abs(y_pred - y_true)) / np.sum(y_true)

# Helper function for Box-Cox and Inverse Box-Cox with a shift to ensure positivity
def arima_boxcox_forecast(series, forecast_periods=6):
    # Shift series if it contains zero or negative values
    if (series <= 0).any():
        shift_val = abs(series.min()) + 1  # Ensure all values are positive
        series = series + shift_val
    else:
        shift_val = 0

    # Apply Box-Cox transformation
    series_boxcox, lam = boxcox(series)
    model = ARIMA(series_boxcox, order=(1, 1, 0))  # ARIMA model
    model_fit = model.fit()
    forecast_boxcox = model_fit.forecast(forecast_periods)
    forecast_arima = inv_boxcox(forecast_boxcox, lam)

    # If a shift was applied, subtract it from the forecasted values
    if shift_val > 0:
        forecast_arima -= shift_val

    return forecast_arima

# Helper function for Holt-Winters Exponential Smoothing
def holt_winters_forecast(series, forecast_periods=6, seasonal_periods=12):
    model = ExponentialSmoothing(
        series, trend='add', seasonal='add', seasonal_periods=seasonal_periods
    ).fit()
    forecast = model.forecast(forecast_periods)
    return forecast

# ARIMA Hyperparameter tuning using auto_arima from pmdarima
def arima_tuning(series):
    model = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=True)
    return model

# Ensemble pipeline using Holt-Winters and ARIMA
def ensemble_pipeline_holt_arima(series, forecast_periods=6, seasonal_periods=12):
    if len(series) < 36:
        return None, None, None, None

    # Split data: last forecast_periods for test
    train_orig = series.iloc[:-forecast_periods]
    test_orig  = series.iloc[-forecast_periods:]

    # ---- Model 1: ARIMA with Box-Cox
    best_arima_model = arima_tuning(train_orig)
    forecast_arima = best_arima_model.predict(n_periods=forecast_periods)

    # ---- Model 2: Holt-Winters Exponential Smoothing
    forecast_hw = holt_winters_forecast(train_orig, forecast_periods=forecast_periods, seasonal_periods=seasonal_periods)

    # ---- Evaluate Ensemble
    y_true = test_orig.values

    # Optimize weight for ensemble (ARIMA + Holt-Winters)
    weights = np.linspace(0, 1, 101)
    best_wmape = np.inf
    best_weight = None
    for w in weights:
        ensemble_pred = w * forecast_arima + (1 - w) * forecast_hw
        wmape_val = compute_wmape(y_true, ensemble_pred)
        if wmape_val < best_wmape:
            best_wmape = wmape_val
            best_weight = w

    return best_wmape, best_weight, forecast_arima, forecast_hw

# =========================== 1. Load & Sort Data =========================== #
file_path = "Series_Hierarchical.csv"  # Update if needed
df = pd.read_csv(file_path)

# Convert to datetime and ensure sorted by date
df['FiscalMonthLastDate'] = pd.to_datetime(df['FiscalMonthLastDate'])
df = df.sort_values('FiscalMonthLastDate').reset_index(drop=True)

# We assume the CSV has "P390_a_x" as the target column
y = df['P390_a_x'].values
dates = df['FiscalMonthLastDate']

# =========================== 2. Train/Test Split =========================== #
train_size = int(0.85 * len(df))  # 85% of data for training
y_train = y[:train_size]
y_test = y[train_size:]
dates_train = dates[:train_size]
dates_test = dates[train_size:]

# =========================== 3. Run the Ensemble Model =========================== #
steps_ahead = len(y_test)  # how many steps to forecast = length of test set
wmape_val, best_weight, forecast_arima, forecast_hw = ensemble_pipeline_holt_arima(df['P390_a_x'], forecast_periods=steps_ahead, seasonal_periods=12)

# Print metrics for the ensemble model
print(f"Best Ensemble Test wMAPE: {wmape_val:.2%} with optimal weight: {best_weight:.2f}")

# =========================== 4. Forecast Next 3 Months =========================== #
steps_future = 3  # Forecast for next 3 months
future_forecast_arima = arima_boxcox_forecast(df['P390_a_x'], forecast_periods=steps_future)
future_forecast_hw = holt_winters_forecast(df['P390_a_x'], forecast_periods=steps_future, seasonal_periods=12)

# Build a date range for the next 3 months
last_date = dates.iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=steps_future, freq='MS')

# =========================== 5. Combine Actual, Test Forecast, and Future Forecast =========================== #
# Store the results in DataFrames
df_results = pd.DataFrame({
    'Date': dates_test,
    'Actual': y_test,
    # 'ARIMA_Forecast': forecast_arima,
    'HW_Forecast': forecast_hw
})

df_future = pd.DataFrame({
    'Date': future_dates,
    # 'ARIMA_Future_Forecast': future_forecast_arima,
    'HW_Future_Forecast': future_forecast_hw
})

# =========================== 6. Plotly Visualization =========================== #
fig = go.Figure()

# Plot the training actual data
fig.add_trace(go.Scatter(
    x=dates_train,
    y=y_train,
    mode='lines+markers',
    name='Train Actual',
    line=dict(color='blue')
))

# Plot the test actual data
fig.add_trace(go.Scatter(
    x=dates_test,
    y=y_test,
    mode='lines+markers',
    name='Test Actual',
    line=dict(color='blue', dash='dot')
))

# Plot the ARIMA test forecast
# fig.add_trace(go.Scatter(
#     x=dates_test,
#     y=forecast_arima,
#     mode='lines+markers',
#     name='ARIMA Test Forecast',
#     line=dict(color='red')
# ))

# Plot the Holt-Winters test forecast
fig.add_trace(go.Scatter(
    x=dates_test,
    y=forecast_hw,
    mode='lines+markers',
    name='HW Test Forecast',
    line=dict(color='green')
))

# Plot the future 3-month forecast
# fig.add_trace(go.Scatter(
#     x=df_future['Date'],
#     y=df_future['ARIMA_Future_Forecast'],
#     mode='lines+markers',
#     name='ARIMA Next 3M Forecast',
#     line=dict(color='red', dash='dot')
# ))

fig.add_trace(go.Scatter(
    x=df_future['Date'],
    y=df_future['HW_Future_Forecast'],
    mode='lines+markers',
    name='HW Next 3M Forecast',
    line=dict(color='green', dash='dot')
))

fig.update_layout(
    title="Holt-Winters Forecast: Test + Next 3 Months",
    xaxis_title="Date",
    yaxis_title="P390_a_x",
    hovermode="x unified"
)

fig.show()

# Display forecast results for the next 3 months
df_future

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[12] intercept   : AIC=inf, Time=3.27 sec
 ARIMA(0,1,0)(0,0,0)[12] intercept   : AIC=535.632, Time=0.02 sec
 ARIMA(1,1,0)(1,0,0)[12] intercept   : AIC=534.870, Time=0.07 sec
 ARIMA(0,1,1)(0,0,1)[12] intercept   : AIC=inf, Time=1.01 sec
 ARIMA(0,1,0)(0,0,0)[12]             : AIC=533.632, Time=0.02 sec
 ARIMA(0,1,0)(1,0,0)[12] intercept   : AIC=536.264, Time=0.17 sec
 ARIMA(0,1,0)(0,0,1)[12] intercept   : AIC=536.672, Time=0.27 sec
 ARIMA(0,1,0)(1,0,1)[12] intercept   : AIC=537.356, Time=1.44 sec
 ARIMA(1,1,0)(0,0,0)[12] intercept   : AIC=533.444, Time=0.03 sec
 ARIMA(1,1,0)(0,0,1)[12] intercept   : AIC=534.950, Time=0.10 sec
 ARIMA(1,1,0)(1,0,1)[12] intercept   : AIC=536.712, Time=0.45 sec
 ARIMA(2,1,0)(0,0,0)[12] intercept   : AIC=529.967, Time=0.17 sec
 ARIMA(2,1,0)(1,0,0)[12] intercept   : AIC=531.935, Time=0.48 sec
 ARIMA(2,1,0)(0,0,1)[12] intercept   : AIC=531.924, Time=0.31 sec
 ARIMA(2,1,0)(1,0,1)[12] intercept   : AI

,Date,HW_Future_Forecast
36,2025-02-01,3281.107911
37,2025-03-01,1546.093586
38,2025-04-01,3204.751019


## 1.3. Dự báo cho series P390 sử dụng Holt winter & Arima

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA


# Helper function for computing wMAPE
def compute_wmape(y_true, y_pred):
    return np.sum(np.abs(y_pred - y_true)) / np.sum(y_true)

# Helper function for Box-Cox and Inverse Box-Cox with a shift to ensure positivity
def arima_boxcox_forecast(series, forecast_periods=6):
    # Shift series if it contains zero or negative values
    if (series <= 0).any():
        shift_val = abs(series.min()) + 1  # Ensure all values are positive
        series = series + shift_val
    else:
        shift_val = 0

    # Apply Box-Cox transformation
    series_boxcox, lam = boxcox(series)
    model = ARIMA(series_boxcox, order=(1, 1, 0))  # ARIMA model
    model_fit = model.fit()
    forecast_boxcox = model_fit.forecast(forecast_periods)
    forecast_arima = inv_boxcox(forecast_boxcox, lam)

    # If a shift was applied, subtract it from the forecasted values
    if shift_val > 0:
        forecast_arima -= shift_val

    return forecast_arima

# Helper function for Holt-Winters Exponential Smoothing
def holt_winters_forecast(series, forecast_periods=6, seasonal_periods=12):
    model = ExponentialSmoothing(
        series, trend='add', seasonal='add', seasonal_periods=seasonal_periods
    ).fit()
    forecast = model.forecast(forecast_periods)
    return forecast

# ARIMA Hyperparameter tuning using auto_arima from pmdarima
def arima_tuning(series):
    model = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=True)
    return model

# Ensemble pipeline using Holt-Winters and ARIMA
def ensemble_pipeline_holt_arima(series, forecast_periods=6, seasonal_periods=12):
    if len(series) < 36:
        return None, None, None, None

    # Split data: last forecast_periods for test
    train_orig = series.iloc[:-forecast_periods]
    test_orig  = series.iloc[-forecast_periods:]

    # ---- Model 1: ARIMA with Box-Cox
    best_arima_model = arima_tuning(train_orig)
    forecast_arima = best_arima_model.predict(n_periods=forecast_periods)

    # ---- Model 2: Holt-Winters Exponential Smoothing
    forecast_hw = holt_winters_forecast(train_orig, forecast_periods=forecast_periods, seasonal_periods=seasonal_periods)

    # ---- Evaluate Ensemble
    y_true = test_orig.values

    # Optimize weight for ensemble (ARIMA + Holt-Winters)
    weights = np.linspace(0, 1, 101)
    best_wmape = np.inf
    best_weight = None
    for w in weights:
        ensemble_pred = w * forecast_arima + (1 - w) * forecast_hw
        wmape_val = compute_wmape(y_true, ensemble_pred)
        if wmape_val < best_wmape:
            best_wmape = wmape_val
            best_weight = w

    return best_wmape, best_weight, forecast_arima, forecast_hw

# =========================== 1. Load & Sort Data =========================== #
file_path = "Series_Hierarchical.csv"  # Update if needed
df = pd.read_csv(file_path)

# Convert to datetime and ensure sorted by date
df['FiscalMonthLastDate'] = pd.to_datetime(df['FiscalMonthLastDate'])
df = df.sort_values('FiscalMonthLastDate').reset_index(drop=True)

# We assume the CSV has "P390_a_x" as the target column
y = df['P390_a_x'].values
dates = df['FiscalMonthLastDate']

# =========================== 2. Train/Test Split =========================== #
train_size = int(0.85 * len(df))  # 85% of data for training
y_train = y[:train_size]
y_test = y[train_size:]
dates_train = dates[:train_size]
dates_test = dates[train_size:]

# =========================== 3. Run the Ensemble Model =========================== #
steps_ahead = len(y_test)  # how many steps to forecast = length of test set
wmape_val, best_weight, forecast_arima, forecast_hw = ensemble_pipeline_holt_arima(df['P390_a_x'], forecast_periods=steps_ahead, seasonal_periods=12)

# Print metrics for the ensemble model
print(f"Best Ensemble Test wMAPE: {wmape_val:.2%} with optimal weight: {best_weight:.2f}")

# =========================== 4. Forecast Next 3 Months =========================== #
steps_future = 3  # Forecast for next 3 months
future_forecast_arima = arima_boxcox_forecast(df['P390_a_x'], forecast_periods=steps_future)
future_forecast_hw = holt_winters_forecast(df['P390_a_x'], forecast_periods=steps_future, seasonal_periods=12)

# Build a date range for the next 3 months
last_date = dates.iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=steps_future, freq='MS')

# =========================== 5. Combine Actual, Test Forecast, and Future Forecast =========================== #
# Store the results in DataFrames
df_results = pd.DataFrame({
    'Date': dates_test,
    'Actual': y_test,
    # 'ARIMA_Forecast': forecast_arima,
    'HW_Forecast': forecast_hw
})

df_future = pd.DataFrame({
    'Date': future_dates,
    # 'ARIMA_Future_Forecast': future_forecast_arima,
    'HW_Future_Forecast': future_forecast_hw
})

# =========================== 6. Plotly Visualization =========================== #
fig = go.Figure()

# Plot the training actual data
fig.add_trace(go.Scatter(
    x=dates_train,
    y=y_train,
    mode='lines+markers',
    name='Train Actual',
    line=dict(color='blue')
))

# Plot the test actual data
fig.add_trace(go.Scatter(
    x=dates_test,
    y=y_test,
    mode='lines+markers',
    name='Test Actual',
    line=dict(color='blue', dash='dot')
))

# Plot the ARIMA test forecast
# fig.add_trace(go.Scatter(
#     x=dates_test,
#     y=forecast_arima,
#     mode='lines+markers',
#     name='ARIMA Test Forecast',
#     line=dict(color='red')
# ))

# Plot the Holt-Winters test forecast
fig.add_trace(go.Scatter(
    x=dates_test,
    y=forecast_hw,
    mode='lines+markers',
    name='HW Test Forecast',
    line=dict(color='green')
))

# Plot the future 3-month forecast
# fig.add_trace(go.Scatter(
#     x=df_future['Date'],
#     y=df_future['ARIMA_Future_Forecast'],
#     mode='lines+markers',
#     name='ARIMA Next 3M Forecast',
#     line=dict(color='red', dash='dot')
# ))

fig.add_trace(go.Scatter(
    x=df_future['Date'],
    y=df_future['HW_Future_Forecast'],
    mode='lines+markers',
    name='HW Next 3M Forecast',
    line=dict(color='green', dash='dot')
))

fig.update_layout(
    title="Holt-Winters Forecast: Test + Next 3 Months",
    xaxis_title="Date",
    yaxis_title="P390_a_x",
    hovermode="x unified"
)

fig.show()

# Display forecast results for the next 3 months
df_future

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[12] intercept   : AIC=inf, Time=1.23 sec
 ARIMA(0,1,0)(0,0,0)[12] intercept   : AIC=535.632, Time=0.03 sec
 ARIMA(1,1,0)(1,0,0)[12] intercept   : AIC=534.870, Time=0.18 sec
 ARIMA(0,1,1)(0,0,1)[12] intercept   : AIC=inf, Time=0.34 sec
 ARIMA(0,1,0)(0,0,0)[12]             : AIC=533.632, Time=0.02 sec
 ARIMA(0,1,0)(1,0,0)[12] intercept   : AIC=536.264, Time=0.13 sec
 ARIMA(0,1,0)(0,0,1)[12] intercept   : AIC=536.672, Time=0.04 sec
 ARIMA(0,1,0)(1,0,1)[12] intercept   : AIC=537.356, Time=0.55 sec
 ARIMA(1,1,0)(0,0,0)[12] intercept   : AIC=533.444, Time=0.03 sec
 ARIMA(1,1,0)(0,0,1)[12] intercept   : AIC=534.950, Time=0.09 sec
 ARIMA(1,1,0)(1,0,1)[12] intercept   : AIC=536.712, Time=0.16 sec
 ARIMA(2,1,0)(0,0,0)[12] intercept   : AIC=529.967, Time=0.12 sec
 ARIMA(2,1,0)(1,0,0)[12] intercept   : AIC=531.935, Time=0.25 sec
 ARIMA(2,1,0)(0,0,1)[12] intercept   : AIC=531.924, Time=0.28 sec
 ARIMA(2,1,0)(1,0,1)[12] intercept   : AI

,Date,HW_Future_Forecast
36,2025-02-01,3281.107911
37,2025-03-01,1546.093586
38,2025-04-01,3204.751019


#### Cải thiện sai số dự báo cho P801_a_x sử dụng LSTM

In [ ]:
import pandas as pd
import numpy as np
import math
from datetime import datetime
import plotly.graph_objs as go

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from skopt import gp_minimize
from skopt.space import Integer, Real

from keras.models import Model
from keras.layers import LSTM, Dense, Input
from keras.optimizers import Adam



file_path = "Series_Hierarchical.csv"  # Update if needed
df = pd.read_csv(file_path)
df['FiscalMonthLastDate'] = pd.to_datetime(df['FiscalMonthLastDate'])
df = df.sort_values('FiscalMonthLastDate').reset_index(drop=True)

# ======================= 2. Add Special Event Flags ======================= #
def add_special_events_features(df):
    df['Christmas']   = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month == 12 else 0)
    df['BlackFriday'] = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month == 11 else 0)
    df['Easter']      = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month in [3, 4] else 0)
    return df

df = add_special_events_features(df)

# ======================= 3. Split into Train & Test ======================= #
train_size = 30
train_df = df.iloc[:train_size]
test_df  = df.iloc[train_size:]

print("Train size:", len(train_df), "Test size:", len(test_df))

# ======================= 4. Prepare Sequences (n_steps=3) ======================= #
n_steps = 3

def prepare_data_multistep(scaled_array, n_steps=3):
    """
    scaled_array: shape (rows, 4) => [OrderQ, Christmas, BlackFri, Easter]
    Return X of shape (samples, n_steps, 4), y of shape (samples,)
    y is the next 'OrderQ' after n_steps.
    """
    X, y = [], []
    for i in range(len(scaled_array) - n_steps):
        seq_x = scaled_array[i : i+n_steps]   # shape (3,4)
        seq_y = scaled_array[i + n_steps, 0]  # col 0 => OrderQ
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

# ======================= 5. Bayesian Optimization ======================= #
def objective(params):
    n_units, learning_rate = params

    # Scale with train only
    scaler = MinMaxScaler()
    train_vals = train_df[['P801_a_x','Christmas','BlackFriday','Easter']].values
    scaled_train = scaler.fit_transform(train_vals)

    test_vals = test_df[['P801_a_x','Christmas','BlackFriday','Easter']].values
    scaled_test = scaler.transform(test_vals)

    # Prepare
    X_train, y_train = prepare_data_multistep(scaled_train, n_steps)
    X_test,  y_test  = prepare_data_multistep(scaled_test,  n_steps)

    # Build LSTM
    inputs = Input(shape=(n_steps, 4))
    x = LSTM(int(n_units), activation='relu', return_sequences=True)(inputs)
    x = LSTM(int(n_units), activation='relu')(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    model.fit(X_train, y_train, epochs=100, verbose=0, validation_data=(X_test, y_test))

    y_pred = model.predict(X_test)
    preds_4cols = np.concatenate([y_pred, np.zeros((y_pred.shape[0], 3))], axis=1)  # pad predictions with 3 zeros
    inv_preds = scaler.inverse_transform(preds_4cols)[:,0]

    # Actual test portion
    actual_vals = test_df['P801_a_x'].values
    actual_for_eval = actual_vals[n_steps:]  # skip first n_steps
    mse = mean_squared_error(actual_for_eval, inv_preds)
    return mse

search_space = [
    Integer(10, 100, name='n_units'),      # 10..100 LSTM units
    Real(1e-4, 1e-2, name='learning_rate') # 1e-4..1e-2
]

result = gp_minimize(objective, search_space, n_calls=15, random_state=42)
best_n_units, best_learning_rate = result.x

print("\n=== Bayesian Optimization Results ===")
print("Best params:", result.x)
print("Best MSE:   ", result.fun)

# ======================= 6. Final Model with Best Params ======================= #
best_n_units = int(best_n_units)  # force int to avoid "output_size must be an integer" error

scaler = MinMaxScaler()
train_vals = train_df[['P801_a_x','Christmas','BlackFriday','Easter']].values
scaled_train = scaler.fit_transform(train_vals)

test_vals = test_df[['P801_a_x','Christmas','BlackFriday','Easter']].values
scaled_test = scaler.transform(test_vals)

X_train, y_train = prepare_data_multistep(scaled_train, n_steps)
X_test,  y_test  = prepare_data_multistep(scaled_test,  n_steps)

inputs = Input(shape=(n_steps, 4))
x = LSTM(best_n_units, activation='relu', return_sequences=True)(inputs)
x = LSTM(best_n_units, activation='relu')(x)
outputs = Dense(1)(x)
model = Model(inputs, outputs)

model.compile(optimizer=Adam(learning_rate=best_learning_rate), loss='mse')
model.fit(X_train, y_train, epochs=150, verbose=0)

# ======================= 7. Evaluate on Test Set ======================= #
y_pred = model.predict(X_test)
preds_4cols = np.concatenate([y_pred, np.zeros((y_pred.shape[0], 3))], axis=1)  # pad predictions with 3 zeros
inv_preds = scaler.inverse_transform(preds_4cols)[:,0]

actual_vals = test_df['P801_a_x'].values
actual_for_eval = actual_vals[n_steps:]  # skip first n_steps


wmape = np.sum(np.abs(actual_for_eval - inv_preds)) / np.sum(np.abs(actual_for_eval)) * 100
r2   = r2_score(actual_for_eval, inv_preds)

print("\n=== Final LSTM Model Metrics on Test Set ===")

print(f"WMAPE: {wmape:.2f}%")

# ======================= 8. Plotly Visualization (Test Forecast) ======================= #
fig = go.Figure()

# Full actual
fig.add_trace(go.Scatter(
    x=df['FiscalMonthLastDate'],
    y=df['P801_a_x'],
    mode='lines+markers',
    name='Actual'
))

test_dates = test_df['FiscalMonthLastDate'].values[n_steps:]
fig.add_trace(go.Scatter(
    x=test_dates,
    y=inv_preds,
    mode='lines+markers',
    name='LSTM Forecast'
))

fig.update_layout(
    title='LSTM Forecast (Test)',
    xaxis_title='Date',
    yaxis_title='P801_a_x',
    hovermode='x unified'
)
fig.show()

# ======================= 9. Predict Next 3 Months ======================= #
# We'll do a step-by-step approach using the last n_steps from the FULL dataset.

all_vals = df[['P801_a_x','Christmas','BlackFriday','Easter']].values
scaled_all = scaler.fit_transform(all_vals)  # re-fit or you can keep the train scaler
seed = scaled_all[-n_steps:]                 # last n_steps from full data

future_steps = 3  # Changed to 3 months forecast
future_preds = []

last_date = df['FiscalMonthLastDate'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=future_steps, freq='MS')

def get_special_flags(date):
    c  = 1 if date.month == 12 else 0
    bf = 1 if date.month == 11 else 0
    e  = 1 if date.month in [3,4] else 0
    return [c, bf, e]

for i in range(future_steps):
    X_input = seed.reshape(1, n_steps, 4)  # shape(1,3,4)
    y_next_scaled = model.predict(X_input)[0][0]  # float

    # invert to actual
    y_next_4cols = np.array([[y_next_scaled, 0, 0, 0]])  # shape(1,4)
    y_next_inv   = scaler.inverse_transform(y_next_4cols)[0, 0]
    future_preds.append(y_next_inv)

    # Build the new row for the next iteration
    next_date = future_dates[i]
    c, bf, e = get_special_flags(next_date)

    # Combine predicted 'OrderQ' with flags -> re-scale
    new_row_4 = np.array([[y_next_inv, c, bf, e]])
    new_row_scaled = scaler.transform(new_row_4)  # shape(1,4)

    # shift seed
    seed = np.concatenate([seed[1:], new_row_scaled], axis=0)

# Create a DataFrame to store the forecast for the next 3 months
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Forecasted P801_a_x': future_preds
})

# Save the forecast DataFrame to a CSV file
forecast_df.to_csv("forecast_next_3_months.csv", index=False)

# Print the DataFrame to the console (optional)
print(forecast_df)

# ======================= Plot the Future 3-Month Forecast ======================= #
future_fig = go.Figure()

future_fig.add_trace(go.Scatter(
    x=df['FiscalMonthLastDate'],
    y=df['P801_a_x'],
    mode='lines+markers',
    name='Historical Actual'
))

future_fig.add_trace(go.Scatter(
    x=future_dates,
    y=future_preds,
    mode='lines+markers',
    name='3-month Future Forecast'
))

future_fig.update_layout(
    title='LSTM Future 3-Month Forecast',
    xaxis_title='Date',
    yaxis_title='P801_a_x',
    hovermode='x unified'
)
future_fig.show()

Train size: 30 Test size: 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step

=== Bayesian Optimization Results ===
Best params: [98, 0.0001]
Best MSE:    410574.46371512744
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step

=== Final LSTM Model Metrics on Test Set ===
WMAPE: 26.39%


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
        Date  Forecasted P801_a_x
0 2025-02-01          3261.028547
1 2025-03-01          3510.931448
2 2025-04-01          3443.613887


# 2. Forecast Model cho Nhóm Y (0.34 ≤ CV < 0.84 – dữ liệu có biến động trung bình)

## 2.1 Xây dựng model dự báo cho dòng series Y bằng Holt winter & Arima

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from pandas.tseries.holiday import USFederalHolidayCalendar
from scipy.stats import boxcox
from statsmodels.tsa.stattools import adfuller
import warnings
import itertools

warnings.filterwarnings("ignore")

# --- 1️⃣ Load dữ liệu ---
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# --- 2️⃣ Lọc các cột thuộc nhóm Y ---
group_y_cols = [col for col in df_final.columns if col.endswith('_y')]
forecast_periods = 6
wmape_threshold = 30  # Mục tiêu: wMAPE < 30%

# --- 3️⃣ Thêm thông tin ngày lễ (Holiday Feature) ---
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=df_final.index.min(), end=df_final.index.max())
df_final["Holiday"] = df_final.index.isin(holidays).astype(int)

# --- 4️⃣ Chạy mô hình dự báo ---
fine_tuned_results = []

print(f"🔄 Bắt đầu Fine-Tuning {len(group_y_cols)} series nhóm Y...")

for col in group_y_cols:
    series = df_final[col].dropna()
    if len(series) < forecast_periods + 1:
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    # Lấy holiday feature
    train_holiday = df_final["Holiday"].iloc[:-forecast_periods]
    test_holiday = df_final["Holiday"].iloc[-forecast_periods:]

    # 🔹 Làm trơn dữ liệu bằng Exponential Moving Average (EMA)
    train_smooth = train.ewm(span=5, adjust=False).mean()

    # 🔹 Loại bỏ outliers bằng IQR
    q1, q3 = np.percentile(train, [25, 75])
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    train_cleaned = train.clip(lower=lower_bound, upper=upper_bound)

    # 🔹 Kiểm tra stationarity (ADF test) để quyết định ARIMA/SARIMA
    adf_pval = adfuller(train_cleaned)[1]
    is_stationary = adf_pval < 0.05

    # 🔹 Tối ưu Holt-Winters với nhiều tham số
    best_wmape = np.inf
    best_model = None
    best_forecast = None
    best_params = None

    hw_configs = list(itertools.product(["add", "mul"], ["add", "mul"], [True, False]))

    for trend, seasonal, damped in hw_configs:
        try:
            model_hw = ExponentialSmoothing(
                train_cleaned, trend=trend, seasonal=seasonal,
                seasonal_periods=12, damped_trend=damped
            ).fit()
            forecast_hw = model_hw.forecast(forecast_periods)
            error_hw = np.sum(np.abs(forecast_hw - test_arr))
            wmape_hw = error_hw / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

            if wmape_hw < best_wmape:
                best_wmape = wmape_hw
                best_model = model_hw
                best_forecast = forecast_hw
                best_params = f"Holt-Winters({trend}, {seasonal}, damped={damped})"
        except:
            continue

    # 🔹 Nếu Holt-Winters không đạt ngưỡng, thử ARIMA
    if best_wmape >= wmape_threshold:
        arima_orders = list(itertools.product([0,1,2], [0,1,2], [0,1,2]))
        for order in arima_orders:
            try:
                model_arima = ARIMA(train_cleaned, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr) * 100

                if wmape_arima < best_wmape:
                    best_wmape = wmape_arima
                    best_model = model_arima
                    best_forecast = forecast_arima
                    best_params = f"ARIMA{order}"
            except:
                continue

    # 🔹 Nếu SARIMA vẫn không đạt, thử XGBoost
    if best_wmape >= wmape_threshold:
        try:
            model_xgb = XGBRegressor(n_estimators=150, learning_rate=0.05)
            model_xgb.fit(np.arange(len(train_cleaned)).reshape(-1,1), train_cleaned)
            forecast_xgb = model_xgb.predict(np.arange(len(train_cleaned), len(train_cleaned) + forecast_periods).reshape(-1,1))
            error_xgb = np.sum(np.abs(forecast_xgb - test_arr))
            wmape_xgb = error_xgb / np.sum(test_arr) * 100

            if wmape_xgb < best_wmape:
                best_wmape = wmape_xgb
                best_model = model_xgb
                best_forecast = forecast_xgb
                best_params = "XGBoost"
        except:
            pass

    fine_tuned_results.append({
        "Series": col, "Best Model": best_params, "New WMAPE": best_wmape,
        "Status": "Improved" if best_wmape < wmape_threshold else "Not Improved"
    })

df_fine_tuned = pd.DataFrame(fine_tuned_results)
num_improved = df_fine_tuned[df_fine_tuned["Status"] == "Improved"].shape[0]

print("\n✅ Số series được cải thiện sau fine-tuning:", num_improved, "/", len(group_y_cols))

# Chuyển đổi danh sách kết quả thành DataFrame
df_fine_tuned = pd.DataFrame(fine_tuned_results)

# Hiển thị danh sách các series đã cải thiện và chưa cải thiện
df_fine_tuned_sorted = df_fine_tuned.sort_values(by=["Status", "New WMAPE"])

# In kết quả
print("\n=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===")
print(df_fine_tuned_sorted.to_string(index=False))


🔄 Bắt đầu Fine-Tuning 73 series nhóm Y...

✅ Số series được cải thiện sau fine-tuning: 25 / 73

=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===
      Series                           Best Model  New WMAPE       Status
    P791_a_y  Holt-Winters(add, mul, damped=True)   9.759663     Improved
    P459_a_y  Holt-Winters(add, add, damped=True)  13.747745     Improved
    P208_b_y Holt-Winters(mul, mul, damped=False)  15.739508     Improved
    P520_b_y Holt-Winters(mul, mul, damped=False)  16.342379     Improved
    P439_a_y                       ARIMA(0, 0, 1)  17.325487     Improved
P210-642_c_y  Holt-Winters(add, add, damped=True)  17.536404     Improved
    P417_b_y Holt-Winters(mul, mul, damped=False)  18.705760     Improved
    P384_b_y Holt-Winters(add, add, damped=False)  19.532592     Improved
P211-632_a_y  Holt-Winters(add, mul, damped=True)  20.361355     Improved
    P358_c_y                       ARIMA(0, 0, 2)  21.082331     Improved
P791-62

In [ ]:
# Chuyển đổi danh sách kết quả thành DataFrame
df_fine_tuned = pd.DataFrame(fine_tuned_results)

# Hiển thị danh sách các series đã cải thiện và chưa cải thiện
df_fine_tuned_sorted = df_fine_tuned.sort_values(by=["Status", "New WMAPE"])

# In kết quả
print("\n=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===")
print(df_fine_tuned_sorted.to_string(index=False))



=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===
      Series                           Best Model  New WMAPE       Status
    P791_a_y  Holt-Winters(add, mul, damped=True)   9.759663     Improved
    P459_a_y  Holt-Winters(add, add, damped=True)  13.747745     Improved
    P208_b_y Holt-Winters(mul, mul, damped=False)  15.739508     Improved
    P520_b_y Holt-Winters(mul, mul, damped=False)  16.342379     Improved
    P439_a_y                       ARIMA(0, 0, 1)  17.325487     Improved
P210-642_c_y  Holt-Winters(add, add, damped=True)  17.536404     Improved
    P417_b_y Holt-Winters(mul, mul, damped=False)  18.705760     Improved
    P384_b_y Holt-Winters(add, add, damped=False)  19.532592     Improved
P211-632_a_y  Holt-Winters(add, mul, damped=True)  20.361355     Improved
    P358_c_y                       ARIMA(0, 0, 2)  21.082331     Improved
P791-625_a_y  Holt-Winters(mul, mul, damped=True)  22.107569     Improved
    P111_b_y                

## 2.2 Cải tiến sai số cho series P297 bằng LSTM với wMAPE của model Hold Winters & ARIMA là  41.676198%

In [ ]:
print("Sai số của series P297_b_y với mô hình ARIMA")
df_fine_tuned_sorted[df_fine_tuned_sorted['Series'] == 'P297_b_y']

In [ ]:
import pandas as pd
import numpy as np
import math
from datetime import datetime
import plotly.graph_objs as go

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from skopt import gp_minimize
from skopt.space import Integer, Real

from keras.models import Model
from keras.layers import LSTM, Dense, Input
from keras.optimizers import Adam



file_path = "Series_Hierarchical.csv"  # Update if needed
df = pd.read_csv(file_path)
df['FiscalMonthLastDate'] = pd.to_datetime(df['FiscalMonthLastDate'])
df = df.sort_values('FiscalMonthLastDate').reset_index(drop=True)

# ======================= 2. Add Special Event Flags ======================= #
def add_special_events_features(df):
    df['Christmas']   = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month == 12 else 0)
    df['BlackFriday'] = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month == 11 else 0)
    df['Easter']      = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month in [3, 4] else 0)
    return df

df = add_special_events_features(df)

# ======================= 3. Split into Train & Test ======================= #
train_size = 32
train_df = df.iloc[:train_size]
test_df  = df.iloc[train_size:]

print("Train size:", len(train_df), "Test size:", len(test_df))

# ======================= 4. Prepare Sequences (n_steps=3) ======================= #
n_steps = 3

def prepare_data_multistep(scaled_array, n_steps=3):
    """
    scaled_array: shape (rows, 4) => [OrderQ, Christmas, BlackFri, Easter]
    Return X of shape (samples, n_steps, 4), y of shape (samples,)
    y is the next 'OrderQ' after n_steps.
    """
    X, y = [], []
    for i in range(len(scaled_array) - n_steps):
        seq_x = scaled_array[i : i+n_steps]   # shape (3,4)
        seq_y = scaled_array[i + n_steps, 0]  # col 0 => OrderQ
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

# ======================= 5. Bayesian Optimization ======================= #
def objective(params):
    n_units, learning_rate = params

    # Scale with train only
    scaler = MinMaxScaler()
    train_vals = train_df[['P297_b_y','Christmas','BlackFriday','Easter']].values
    scaled_train = scaler.fit_transform(train_vals)

    test_vals = test_df[['P297_b_y','Christmas','BlackFriday','Easter']].values
    scaled_test = scaler.transform(test_vals)

    # Prepare
    X_train, y_train = prepare_data_multistep(scaled_train, n_steps)
    X_test,  y_test  = prepare_data_multistep(scaled_test,  n_steps)

    # Build LSTM
    inputs = Input(shape=(n_steps, 4))
    x = LSTM(int(n_units), activation='relu', return_sequences=True)(inputs)
    x = LSTM(int(n_units), activation='relu')(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    model.fit(X_train, y_train, epochs=100, verbose=0, validation_data=(X_test, y_test))

    y_pred = model.predict(X_test)
    preds_4cols = np.concatenate([y_pred, np.zeros((y_pred.shape[0], 3))], axis=1)  # pad predictions with 3 zeros
    inv_preds = scaler.inverse_transform(preds_4cols)[:,0]

    # Actual test portion
    actual_vals = test_df['P297_b_y'].values
    actual_for_eval = actual_vals[n_steps:]  # skip first n_steps
    mse = mean_squared_error(actual_for_eval, inv_preds)
    return mse

search_space = [
    Integer(10, 100, name='n_units'),      # 10..100 LSTM units
    Real(1e-4, 1e-2, name='learning_rate') # 1e-4..1e-2
]

result = gp_minimize(objective, search_space, n_calls=15, random_state=42)
best_n_units, best_learning_rate = result.x

print("\n=== Bayesian Optimization Results ===")
print("Best params:", result.x)
print("Best MSE:   ", result.fun)

# ======================= 6. Final Model with Best Params ======================= #
best_n_units = int(best_n_units)  # force int to avoid "output_size must be an integer" error

scaler = MinMaxScaler()
train_vals = train_df[['P297_b_y','Christmas','BlackFriday','Easter']].values
scaled_train = scaler.fit_transform(train_vals)

test_vals = test_df[['P297_b_y','Christmas','BlackFriday','Easter']].values
scaled_test = scaler.transform(test_vals)

X_train, y_train = prepare_data_multistep(scaled_train, n_steps)
X_test,  y_test  = prepare_data_multistep(scaled_test,  n_steps)

inputs = Input(shape=(n_steps, 4))
x = LSTM(best_n_units, activation='relu', return_sequences=True)(inputs)
x = LSTM(best_n_units, activation='relu')(x)
outputs = Dense(1)(x)
model = Model(inputs, outputs)

model.compile(optimizer=Adam(learning_rate=best_learning_rate), loss='mse')
model.fit(X_train, y_train, epochs=150, verbose=0)

# ======================= 7. Evaluate on Test Set ======================= #
y_pred = model.predict(X_test)
preds_4cols = np.concatenate([y_pred, np.zeros((y_pred.shape[0], 3))], axis=1)  # pad predictions with 3 zeros
inv_preds = scaler.inverse_transform(preds_4cols)[:,0]

actual_vals = test_df['P297_b_y'].values
actual_for_eval = actual_vals[n_steps:]  # skip first n_steps


wmape = np.sum(np.abs(actual_for_eval - inv_preds)) / np.sum(np.abs(actual_for_eval)) * 100
r2   = r2_score(actual_for_eval, inv_preds)

print("\n=== Final LSTM Model Metrics on Test Set ===")

print(f"WMAPE: {wmape:.2f}%")

# ======================= 8. Plotly Visualization (Test Forecast) ======================= #
fig = go.Figure()

# Full actual
fig.add_trace(go.Scatter(
    x=df['FiscalMonthLastDate'],
    y=df['P297_b_y'],
    mode='lines+markers',
    name='Actual'
))

test_dates = test_df['FiscalMonthLastDate'].values[n_steps:]
fig.add_trace(go.Scatter(
    x=test_dates,
    y=inv_preds,
    mode='lines+markers',
    name='LSTM Forecast'
))

fig.update_layout(
    title='LSTM Forecast (Test)',
    xaxis_title='Date',
    yaxis_title='P297_b_y',
    hovermode='x unified'
)
fig.show()

# ======================= 9. Predict Next 3 Months ======================= #
# We'll do a step-by-step approach using the last n_steps from the FULL dataset.

all_vals = df[['P297_b_y','Christmas','BlackFriday','Easter']].values
scaled_all = scaler.fit_transform(all_vals)  # re-fit or you can keep the train scaler
seed = scaled_all[-n_steps:]                 # last n_steps from full data

future_steps = 3  # Changed to 3 months forecast
future_preds = []

last_date = df['FiscalMonthLastDate'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=future_steps, freq='MS')

def get_special_flags(date):
    c  = 1 if date.month == 12 else 0
    bf = 1 if date.month == 11 else 0
    e  = 1 if date.month in [3,4] else 0
    return [c, bf, e]

for i in range(future_steps):
    X_input = seed.reshape(1, n_steps, 4)  # shape(1,3,4)
    y_next_scaled = model.predict(X_input)[0][0]  # float

    # invert to actual
    y_next_4cols = np.array([[y_next_scaled, 0, 0, 0]])  # shape(1,4)
    y_next_inv   = scaler.inverse_transform(y_next_4cols)[0, 0]
    future_preds.append(y_next_inv)

    # Build the new row for the next iteration
    next_date = future_dates[i]
    c, bf, e = get_special_flags(next_date)

    # Combine predicted 'OrderQ' with flags -> re-scale
    new_row_4 = np.array([[y_next_inv, c, bf, e]])
    new_row_scaled = scaler.transform(new_row_4)  # shape(1,4)

    # shift seed
    seed = np.concatenate([seed[1:], new_row_scaled], axis=0)

# Create a DataFrame to store the forecast for the next 3 months
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Forecasted P297_b_y': future_preds
})

# Save the forecast DataFrame to a CSV file
forecast_df.to_csv("forecast_next_3_months.csv", index=False)

# Print the DataFrame to the console (optional)
print(forecast_df)

# ======================= Plot the Future 3-Month Forecast ======================= #
future_fig = go.Figure()

future_fig.add_trace(go.Scatter(
    x=df['FiscalMonthLastDate'],
    y=df['P297_b_y'],
    mode='lines+markers',
    name='Historical Actual'
))

future_fig.add_trace(go.Scatter(
    x=future_dates,
    y=future_preds,
    mode='lines+markers',
    name='3-month Future Forecast'
))

future_fig.update_layout(
    title='LSTM Future 3-Month Forecast',
    xaxis_title='Date',
    yaxis_title='P297_b_y',
    hovermode='x unified'
)
future_fig.show()

Train size: 32 Test size: 4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 350ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step

=== Bayesian Optimization Results ===
Best params: [np.int64(61), 0.0013512622809612256]
Best MSE:    5047.794975682409
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step

=== Final LSTM Model Metrics on Test Set ===
WMAPE: 19.10%


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
        Date  Forecasted P297_b_y
0 2025-02-01           695.168806
1 2025-03-01           331.331083
2 2025-04-01           636.160257


# 3. Forecast Model cho Nhóm Z (CV >= 0.84 – dữ liệu có biến động cao)

## 3.1 Xây dựng model dự báo cho dòng series Z bằng SARIMAX, Gradient Boosting Regressor, Ridge Regression, and Naïve Forecast

In [ ]:
# !pip install --upgrade numpy
# !pip install --upgrade jax

In [ ]:
import warnings
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.stats import mstats, boxcox
from scipy.special import inv_boxcox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

#############################################
# Helper Functions
#############################################
def compute_wmape(y_true, y_pred):
    total = np.sum(np.abs(y_true))
    return np.sum(np.abs(y_pred - y_true)) / total if total != 0 else 0.0

def is_holiday(date):
    return 1 if date.month in {1,7,12} else 0

def create_features_log(series, lag, fourier_harmonics=3):
    df_feat = pd.DataFrame({'y': series})
    for i in range(1, lag+1):
        df_feat[f'lag_{i}'] = series.shift(i)
    df_feat['month'] = df_feat.index.month
    df_feat['quarter'] = df_feat.index.quarter
    df_feat['year'] = df_feat.index.year
    df_feat['time_index'] = np.arange(len(df_feat))
    period = 12
    t = np.arange(len(df_feat))
    for k in range(1, fourier_harmonics+1):
        df_feat[f'sin_{k}'] = np.sin(2*np.pi*k*t/period)
        df_feat[f'cos_{k}'] = np.cos(2*np.pi*k*t/period)
    df_feat['holiday'] = df_feat.index.to_series().apply(is_holiday)
    df_feat.fillna(method='ffill', inplace=True)
    df_feat.fillna(method='bfill', inplace=True)
    return df_feat

def process_series(series):
    series_w = pd.Series(mstats.winsorize(series, limits=[0.05,0.05]), index=series.index)
    series_s = series_w.rolling(window=5, center=True).median()
    series_s = series_s.fillna(method='bfill').fillna(method='ffill')
    use_shift = False
    if (series_s <= 0).any():
        shift_val = abs(series_s.min()) + 1
        series_s = series_s + shift_val
        use_shift = True
    else:
        shift_val = 0
    series_log = np.log1p(series_s)
    return series_s, series_log, use_shift, shift_val

def is_intermittent(series, threshold=0.8):
    return np.mean(series == 0) >= threshold

def croston_forecast(x, h, alpha=0.1):
    x = np.asarray(x)
    if np.all(x == 0):
        return np.zeros(h)
    demand = []
    intervals = []
    last_index = 0
    for t in range(len(x)):
        if x[t] > 0:
            demand.append(x[t])
            intervals.append(t - last_index + 1)
            last_index = t + 1
    d_hat = demand[0]
    p_hat = intervals[0]
    for i in range(1, len(demand)):
        d_hat = alpha * demand[i] + (1 - alpha) * d_hat
        p_hat = alpha * intervals[i] + (1 - alpha) * p_hat
    return np.full(h, d_hat / p_hat)

#############################################
# New Ensemble Pipeline for Group Z using Four Models & Median Ensemble
#############################################
def ensemble_pipeline_z_final(series, forecast_periods=6, lag=12, alpha_croston=0.1):
    # Preprocess series
    series_orig, series_log, use_shift, shift_val = process_series(series)
    if len(series_log) < 36:
        return None
    train_orig = series_orig.iloc[:-forecast_periods]
    test_orig  = series_orig.iloc[-forecast_periods:]

    # If highly intermittent, use Croston's method directly:
    if is_intermittent(train_orig, threshold=0.8):
        forecast_model = croston_forecast(train_orig.values, forecast_periods, alpha=alpha_croston)
        if use_shift:
            forecast_model = forecast_model - shift_val
        return compute_wmape(test_orig.values, forecast_model)

    # Model 1: SARIMAX + Box-Cox on original processed series
    try:
        if np.allclose(train_orig.values, train_orig.values[0]):
            forecast_model1 = np.full(forecast_periods, train_orig.values[0])
        else:
            train_boxcox, lam = boxcox(train_orig)
            order = (1,1,0)
            seasonal_order = (1,0,1,12)
            model_sarimax = SARIMAX(train_boxcox, order=order, seasonal_order=seasonal_order).fit(disp=False)
            forecast_boxcox = model_sarimax.forecast(forecast_periods)
            forecast_model1 = inv_boxcox(forecast_boxcox, lam)
            med = np.median(train_orig)
            if med > 0 and (np.any(forecast_model1 > 2 * med) or np.any(forecast_model1 < 0) or not np.all(np.isfinite(forecast_model1))):
                forecast_model1 = np.full(forecast_periods, med)
    except Exception as e:
        forecast_model1 = np.full(forecast_periods, np.median(train_orig))
    if use_shift:
        forecast_model1 = forecast_model1 - shift_val

    # Model 2: LightGBM on extended features
    df_feat = create_features_log(series_log, lag, fourier_harmonics=3)
    train_feat = df_feat.iloc[:-forecast_periods]
    test_feat = df_feat.iloc[-forecast_periods:]
    X_train = train_feat.drop(columns=['y'])
    y_train = train_feat['y']
    X_test = test_feat.drop(columns=['y'])
    from sklearn.ensemble import GradientBoostingRegressor
    model_gbr = GradientBoostingRegressor(learning_rate=0.01, n_estimators=200, max_depth=3, random_state=42)
    model_gbr.fit(X_train, y_train)
    y_pred_log_gbr = model_gbr.predict(X_test)
    forecast_model2 = np.expm1(y_pred_log_gbr)

    # Model 3: Ridge Regression on extended features
    from sklearn.linear_model import Ridge
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_train, y_train)
    y_pred_log_ridge = model_ridge.predict(X_test)
    forecast_model3 = np.expm1(y_pred_log_ridge)

    # Model 4: Naïve forecast using last observed value
    forecast_model4 = np.full(forecast_periods, train_orig.iloc[-1])

    # Combine forecasts via elementwise median across the four models
    forecasts = np.vstack([forecast_model1, forecast_model2, forecast_model3, forecast_model4])
    ensemble_pred = np.median(forecasts, axis=0)

    y_true = test_orig.values
    return compute_wmape(y_true, ensemble_pred)

#############################################
# MAIN: Evaluate the final ensemble pipeline on each series in Group Z
#############################################
df = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)
group_z_cols = [col for col in df.columns if col != 'Total' and col.endswith('_z')]

results = []
for col in group_z_cols:
    s = df[col]
    wmape_val = ensemble_pipeline_z_final(s, forecast_periods=6, lag=12, alpha_croston=0.1)
    results.append({"Series": col, "Ensemble Test wMAPE": wmape_val})

results_df = pd.DataFrame(results)
results_df["Ensemble Test wMAPE"] = results_df["Ensemble Test wMAPE"].apply(lambda x: "{:.2%}".format(x) if pd.notnull(x) else None)
print("\n----- Final Ensemble wMAPE for Each Series in Group Z -----")
print(results_df)

def parse_percent(pct_str):
    return float(pct_str.strip('%'))/100 if pct_str is not None else None

results_df["wMAPE_float"] = results_df["Ensemble Test wMAPE"].apply(parse_percent)
valid_count = results_df['wMAPE_float'].dropna().apply(lambda x: x < 0.35).sum()
total_series = results_df['wMAPE_float'].dropna().shape[0]
print(f"\nOut of {total_series} series in Group Z, {valid_count} have Ensemble Test wMAPE below 35%.")

/usr/local/lib/python3.11/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.5.3, so it will not be used.
  warnings.warn(



----- Final Ensemble wMAPE for Each Series in Group Z -----
           Series Ensemble Test wMAPE
0    P008-813_c_z              37.83%
1    P009-813_b_z              97.66%
2    P011-776_c_z             226.08%
3        P013_a_z             113.02%
4    P014-813_b_z              63.82%
..            ...                 ...
126     PA784_c_z              25.64%
127     PA792_c_z              10.64%
128     PA798_c_z              26.18%
129     PA802_c_z              41.72%
130     PA804_c_z              64.51%

[131 rows x 2 columns]

Out of 131 series in Group Z, 44 have Ensemble Test wMAPE below 35%.


## 3.2. Dự báo cho series PA802 với SARIMAX, Gradient Boosting Regressor, Ridge Regression, and Naïve Forecast

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score

# For SARIMAX (SARIMA model)
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.stats import boxcox, mstats
from scipy.special import inv_boxcox

# For LightGBM model
import lightgbm as lgb

# For Ridge Regression model
from sklearn.linear_model import Ridge

# For Croston's method
def croston_forecast(x, h, alpha=0.1):
    x = np.asarray(x)
    if np.all(x == 0):
        return np.zeros(h)
    demand = []
    intervals = []
    last_index = 0
    for t in range(len(x)):
        if x[t] > 0:
            demand.append(x[t])
            intervals.append(t - last_index + 1)
            last_index = t + 1
    d_hat = demand[0]
    p_hat = intervals[0]
    for i in range(1, len(demand)):
        d_hat = alpha * demand[i] + (1 - alpha) * d_hat
        p_hat = alpha * intervals[i] + (1 - alpha) * p_hat
    return np.full(h, d_hat / p_hat)

# =========================== 1. Load & Sort Data =========================== #
file_path = "Series_Hierarchical.csv"  # Update if needed
df = pd.read_csv(file_path)

# Convert to datetime and sort
df['FiscalMonthLastDate'] = pd.to_datetime(df['FiscalMonthLastDate'])
df = df.sort_values('FiscalMonthLastDate').reset_index(drop=True)

# Create a Series with a datetime index for proper feature creation
y_series = pd.Series(df['PA802_c_z'].values, index=df['FiscalMonthLastDate'])

# =========================== 2. Train/Test Split =========================== #
train_size = int(0.85 * len(df))
y_train = y_series.iloc[:train_size]
y_test = y_series.iloc[train_size:]
dates_train = y_train.index
dates_test = y_test.index

# =========================== Helper Functions =========================== #
def is_holiday(date):
    return 1 if date.month in {1, 7, 12} else 0

def create_features_log(series, lag, fourier_harmonics=3):
    df_feat = pd.DataFrame({'y': series})
    for i in range(1, lag + 1):
        df_feat[f'lag_{i}'] = series.shift(i)
    df_feat['month'] = df_feat.index.month
    df_feat['quarter'] = df_feat.index.quarter
    df_feat['year'] = df_feat.index.year
    df_feat['time_index'] = np.arange(len(df_feat))
    period = 12
    t = np.arange(len(df_feat))
    for k in range(1, fourier_harmonics + 1):
        df_feat[f'sin_{k}'] = np.sin(2 * np.pi * k * t / period)
        df_feat[f'cos_{k}'] = np.cos(2 * np.pi * k * t / period)
    df_feat['holiday'] = df_feat.index.to_series().apply(is_holiday)
    df_feat.fillna(method='ffill', inplace=True)
    df_feat.fillna(method='bfill', inplace=True)
    # Drop the first `lag` rows which may contain imputed values
    df_feat = df_feat.iloc[lag:]
    return df_feat

def process_series(series):
    if isinstance(series, np.ndarray):
        series = pd.Series(series)
    series_w = pd.Series(mstats.winsorize(series, limits=[0.05, 0.05]), index=series.index)
    series_s = series_w.rolling(window=5, center=True).median()
    series_s = series_s.fillna(method='bfill').fillna(method='ffill')
    use_shift = False
    if (series_s <= 0).any():
        shift_val = abs(series_s.min()) + 1
        series_s = series_s + shift_val
        use_shift = True
    else:
        shift_val = 0
    series_log = np.log1p(series_s)
    return series_s, series_log, use_shift, shift_val

def is_intermittent(series, threshold=0.8):
    return np.mean(series == 0) >= threshold

def compute_wmape(y_true, y_pred):
    total = np.sum(np.abs(y_true))
    return np.sum(np.abs(y_pred - y_true)) / total if total != 0 else 0.0

# =========================== Forecasting Functions =========================== #
def forecast_sarimax(series, forecast_periods):
    series_orig, series_log, use_shift, shift_val = process_series(series)
    if len(series_log) < 36:
        return None  # Not enough data for SARIMAX
    train_orig = series_orig.iloc[:-forecast_periods]
    train_boxcox, lam = boxcox(train_orig)
    order = (1, 1, 0)
    seasonal_order = (1, 0, 1, 12)
    model_sarimax = SARIMAX(train_boxcox, order=order, seasonal_order=seasonal_order).fit(disp=False)
    forecast_boxcox = model_sarimax.forecast(forecast_periods)
    forecast_model1 = inv_boxcox(forecast_boxcox, lam)
    return np.asarray(forecast_model1).flatten()

def forecast_ridge(series, forecast_periods, lag=12):
    series_s, series_log, use_shift, shift_val = process_series(series)
    df_feat = create_features_log(series_log, lag)
    train_feat = df_feat.iloc[:-forecast_periods]
    test_feat = df_feat.iloc[-forecast_periods:]
    X_train = train_feat.drop(columns=['y'])
    y_train_feat = train_feat['y']
    X_test = test_feat.drop(columns=['y'])
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_train, y_train_feat)
    y_pred_log_ridge = model_ridge.predict(X_test)
    return np.asarray(np.expm1(y_pred_log_ridge)).flatten()

def forecast_croston(series, forecast_periods):
    return np.asarray(croston_forecast(series.values, forecast_periods)).flatten()

# =========================== 4. Forecast on Test (In-Sample) =========================== #
steps_ahead = len(y_test)
forecast_sarimax_result = forecast_sarimax(y_train, steps_ahead)
forecast_ridge_result = forecast_ridge(y_train, steps_ahead)
forecast_croston_result = forecast_croston(y_train, steps_ahead)

# Combine available forecasts (skip SARIMAX if None)
forecasts = []
if forecast_sarimax_result is not None:
    forecasts.append(forecast_sarimax_result)
else:
    print("SARIMAX forecast is None (insufficient training data).")
forecasts.append(forecast_ridge_result)
forecasts.append(forecast_croston_result)

ensemble_pred = np.median(np.vstack(forecasts), axis=0)

# =========================== 5. Calculate Metrics =========================== #
print("=== Ensemble Model Metrics ===")
print(f"WMAPE: {compute_wmape(y_test, ensemble_pred):.2%}")

# =========================== 6. Forecast Next 3 Months (Future Forecast) =========================== #
forecast_sarimax_result_full = forecast_sarimax(y_series, 3)
forecast_ridge_result_full = forecast_ridge(y_series, 3)
forecast_croston_result_full = forecast_croston(y_series, 3)

forecasts_future = []
if forecast_sarimax_result_full is not None:
    forecasts_future.append(forecast_sarimax_result_full)
forecasts_future.append(forecast_ridge_result_full)
forecasts_future.append(forecast_croston_result_full)

ensemble_pred_future = np.median(np.vstack(forecasts_future), axis=0)

# Create a date range for the next 3 months
last_date = y_series.index[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=3, freq='MS')

# Print future forecast values
print("=== Future 3-Month Ensemble Forecast ===")
print(pd.DataFrame({'Date': future_dates, 'Forecast': ensemble_pred_future}))

# =========================== 7. Plot Results =========================== #
fig = go.Figure()

# Plot training data
fig.add_trace(go.Scatter(
    x=dates_train,
    y=y_train,
    mode='lines+markers',
    name='Train Actual',
    line=dict(color='blue')
))

# Plot test data
fig.add_trace(go.Scatter(
    x=dates_test,
    y=y_test,
    mode='lines+markers',
    name='Test Actual',
    line=dict(color='blue', dash='dot')
))

# Plot ensemble forecast for the test period
fig.add_trace(go.Scatter(
    x=dates_test,
    y=ensemble_pred,
    mode='lines+markers',
    name='Ensemble Forecast (Test)',
    line=dict(color='red')
))

# Plot ensemble forecast for the next 3 months
fig.add_trace(go.Scatter(
    x=future_dates,
    y=ensemble_pred_future,
    mode='lines+markers',
    name='Future 3M Forecast',
    line=dict(color='green')
))

fig.update_layout(
    title="Ensemble Model Forecast: Historical, Test, & Future 3 Months",
    xaxis_title="Date",
    yaxis_title="PA802_c_z",
    hovermode="x unified"
)

fig.show()

SARIMAX forecast is None (insufficient training data).
=== Ensemble Model Metrics ===
WMAPE: 131.88%
=== Future 3-Month Ensemble Forecast ===
        Date   Forecast
0 2025-02-01  12.917594
1 2025-03-01  11.014718
2 2025-04-01  10.038862


## 3.3 Cải thiện dự báo cho series PA802 với LSTM

In [ ]:
import pandas as pd
import numpy as np
import math
from datetime import datetime
import plotly.graph_objs as go

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from skopt import gp_minimize
from skopt.space import Integer, Real

from keras.models import Model
from keras.layers import LSTM, Dense, Input
from keras.optimizers import Adam



file_path = "Series_Hierarchical.csv"  # Update if needed
df = pd.read_csv(file_path)
df['FiscalMonthLastDate'] = pd.to_datetime(df['FiscalMonthLastDate'])
df = df.sort_values('FiscalMonthLastDate').reset_index(drop=True)

# ======================= 2. Add Special Event Flags ======================= #
def add_special_events_features(df):
    df['Christmas']   = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month == 12 else 0)
    df['BlackFriday'] = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month == 11 else 0)
    df['Easter']      = df['FiscalMonthLastDate'].apply(lambda x: 1 if x.month in [3, 4] else 0)
    return df

df = add_special_events_features(df)

# ======================= 3. Split into Train & Test ======================= #
train_size = 32
train_df = df.iloc[:train_size]
test_df  = df.iloc[train_size:]

print("Train size:", len(train_df), "Test size:", len(test_df))

# ======================= 4. Prepare Sequences (n_steps=3) ======================= #
n_steps = 3

def prepare_data_multistep(scaled_array, n_steps=3):
    """
    scaled_array: shape (rows, 4) => [OrderQ, Christmas, BlackFri, Easter]
    Return X of shape (samples, n_steps, 4), y of shape (samples,)
    y is the next 'OrderQ' after n_steps.
    """
    X, y = [], []
    for i in range(len(scaled_array) - n_steps):
        seq_x = scaled_array[i : i+n_steps]   # shape (3,4)
        seq_y = scaled_array[i + n_steps, 0]  # col 0 => OrderQ
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

# ======================= 5. Bayesian Optimization ======================= #
def objective(params):
    n_units, learning_rate = params

    # Scale with train only
    scaler = MinMaxScaler()
    train_vals = train_df[['PA802_c_z','Christmas','BlackFriday','Easter']].values
    scaled_train = scaler.fit_transform(train_vals)

    test_vals = test_df[['PA802_c_z','Christmas','BlackFriday','Easter']].values
    scaled_test = scaler.transform(test_vals)

    # Prepare
    X_train, y_train = prepare_data_multistep(scaled_train, n_steps)
    X_test,  y_test  = prepare_data_multistep(scaled_test,  n_steps)

    # Build LSTM
    inputs = Input(shape=(n_steps, 4))
    x = LSTM(int(n_units), activation='relu', return_sequences=True)(inputs)
    x = LSTM(int(n_units), activation='relu')(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    model.fit(X_train, y_train, epochs=200, verbose=0, validation_data=(X_test, y_test))

    y_pred = model.predict(X_test)
    preds_4cols = np.concatenate([y_pred, np.zeros((y_pred.shape[0], 3))], axis=1)  # pad predictions with 3 zeros
    inv_preds = scaler.inverse_transform(preds_4cols)[:,0]

    # Actual test portion
    actual_vals = test_df['PA802_c_z'].values
    actual_for_eval = actual_vals[n_steps:]  # skip first n_steps
    mse = mean_squared_error(actual_for_eval, inv_preds)
    return mse

search_space = [
    Integer(10, 100, name='n_units'),      # 10..100 LSTM units
    Real(1e-4, 1e-2, name='learning_rate') # 1e-4..1e-2
]

result = gp_minimize(objective, search_space, n_calls=15, random_state=42)
best_n_units, best_learning_rate = result.x

print("\n=== Bayesian Optimization Results ===")
print("Best params:", result.x)
print("Best MSE:   ", result.fun)

# ======================= 6. Final Model with Best Params ======================= #
best_n_units = int(best_n_units)  # force int to avoid "output_size must be an integer" error

scaler = MinMaxScaler()
train_vals = train_df[['PA802_c_z','Christmas','BlackFriday','Easter']].values
scaled_train = scaler.fit_transform(train_vals)

test_vals = test_df[['PA802_c_z','Christmas','BlackFriday','Easter']].values
scaled_test = scaler.transform(test_vals)

X_train, y_train = prepare_data_multistep(scaled_train, n_steps)
X_test,  y_test  = prepare_data_multistep(scaled_test,  n_steps)

inputs = Input(shape=(n_steps, 4))
x = LSTM(best_n_units, activation='relu', return_sequences=True)(inputs)
x = LSTM(best_n_units, activation='relu')(x)
outputs = Dense(1)(x)
model = Model(inputs, outputs)

model.compile(optimizer=Adam(learning_rate=best_learning_rate), loss='mse')
model.fit(X_train, y_train, epochs=150, verbose=0)

# ======================= 7. Evaluate on Test Set ======================= #
y_pred = model.predict(X_test)
preds_4cols = np.concatenate([y_pred, np.zeros((y_pred.shape[0], 3))], axis=1)  # pad predictions with 3 zeros
inv_preds = scaler.inverse_transform(preds_4cols)[:,0]

actual_vals = test_df['PA802_c_z'].values
actual_for_eval = actual_vals[n_steps:]  # skip first n_steps


wmape = np.sum(np.abs(actual_for_eval - inv_preds)) / np.sum(np.abs(actual_for_eval)) * 100
r2   = r2_score(actual_for_eval, inv_preds)

print("\n=== Final LSTM Model Metrics on Test Set ===")

print(f"WMAPE: {wmape:.2f}%")

# ======================= 8. Plotly Visualization (Test Forecast) ======================= #
fig = go.Figure()

# Full actual
fig.add_trace(go.Scatter(
    x=df['FiscalMonthLastDate'],
    y=df['PA802_c_z'],
    mode='lines+markers',
    name='Actual'
))

test_dates = test_df['FiscalMonthLastDate'].values[n_steps:]
fig.add_trace(go.Scatter(
    x=test_dates,
    y=inv_preds,
    mode='lines+markers',
    name='LSTM Forecast'
))

fig.update_layout(
    title='LSTM Forecast (Test)',
    xaxis_title='Date',
    yaxis_title='PA802_c_z',
    hovermode='x unified'
)
fig.show()

# ======================= 9. Predict Next 3 Months ======================= #
# We'll do a step-by-step approach using the last n_steps from the FULL dataset.

all_vals = df[['PA802_c_z','Christmas','BlackFriday','Easter']].values
scaled_all = scaler.fit_transform(all_vals)  # re-fit or you can keep the train scaler
seed = scaled_all[-n_steps:]                 # last n_steps from full data

future_steps = 3  # Changed to 3 months forecast
future_preds = []

last_date = df['FiscalMonthLastDate'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=future_steps, freq='MS')

def get_special_flags(date):
    c  = 1 if date.month == 12 else 0
    bf = 1 if date.month == 11 else 0
    e  = 1 if date.month in [3,4] else 0
    return [c, bf, e]

for i in range(future_steps):
    X_input = seed.reshape(1, n_steps, 4)  # shape(1,3,4)
    y_next_scaled = model.predict(X_input)[0][0]  # float

    # invert to actual
    y_next_4cols = np.array([[y_next_scaled, 0, 0, 0]])  # shape(1,4)
    y_next_inv   = scaler.inverse_transform(y_next_4cols)[0, 0]
    future_preds.append(y_next_inv)

    # Build the new row for the next iteration
    next_date = future_dates[i]
    c, bf, e = get_special_flags(next_date)

    # Combine predicted 'OrderQ' with flags -> re-scale
    new_row_4 = np.array([[y_next_inv, c, bf, e]])
    new_row_scaled = scaler.transform(new_row_4)  # shape(1,4)

    # shift seed
    seed = np.concatenate([seed[1:], new_row_scaled], axis=0)

# Create a DataFrame to store the forecast for the next 3 months
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Forecasted PA802_c_z': future_preds
})

# Save the forecast DataFrame to a CSV file
forecast_df.to_csv("forecast_next_3_months.csv", index=False)

# Print the DataFrame to the console (optional)
print(forecast_df)

# ======================= Plot the Future 3-Month Forecast ======================= #
future_fig = go.Figure()

future_fig.add_trace(go.Scatter(
    x=df['FiscalMonthLastDate'],
    y=df['PA802_c_z'],
    mode='lines+markers',
    name='Historical Actual'
))

future_fig.add_trace(go.Scatter(
    x=future_dates,
    y=future_preds,
    mode='lines+markers',
    name='3-month Future Forecast'
))

future_fig.update_layout(
    title='LSTM Future 3-Month Forecast',
    xaxis_title='Date',
    yaxis_title='PA802_c_z',
    hovermode='x unified'
)
future_fig.show()

Train size: 32 Test size: 4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 464ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step

=== Bayesian Optimization Results ===
Best params: [np.int64(86), 0.0016500941408980239]
Best MSE:    6.381622278127352
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step

=== Final LSTM Model Metrics on Test Set ===
WMAPE: 159.89%


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
        Date  Forecasted PA802_c_z
0 2025-02-01             10.280969
1 2025-03-01              5.944609
2 2025-04-01             40.241340


# Phase 2: Deploy model for forecasting

## 1. Triển khai model dự báo cho dòng series X với gradio

In [ ]:
!pip install gradio
!pip uninstall pmdarima
!pip install --no-cache-dir pmdarima
!pip install numpy==1.25
!pip install --upgrade gradio
!python app.py

Found existing installation: pmdarima 2.0.4
Uninstalling pmdarima-2.0.4:
  Would remove:
    /usr/local/lib/python3.11/dist-packages/pmdarima-2.0.4.dist-info/*
    /usr/local/lib/python3.11/dist-packages/pmdarima/*
Proceed (Y/n)? y
  Successfully uninstalled pmdarima-2.0.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 27.7 MB/s eta 0:00:00


ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# Modeling
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Deep learning
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Gradio
import gradio as gr

#########################################
# 1. ĐỌC & CHUẨN BỊ DỮ LIỆU
#########################################
df = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)

group_x_cols = [col for col in df.columns if col.endswith('_x')]
group_y_cols = [col for col in df.columns if col.endswith('_y')]
group_z_cols = [col for col in df.columns if col.endswith('_z')]

# Thêm "All" vào dropdown
group_x_options = ["All"] + group_x_cols
group_y_options = ["All"] + group_y_cols
group_z_options = ["All"] + group_z_cols

#########################################
# 2. HÀM TIỆN ÍCH CHUNG
#########################################
def build_future_dates(last_date, periods):
    """Tạo dãy datetime hàng tháng, bắt đầu từ tháng tiếp theo của last_date."""
    return pd.date_range(start=last_date + pd.DateOffset(months=1),
                         periods=periods, freq="M")

def plot_history_and_forecast(history_series, forecast_df, title):
    """Trả về plotly figure hiển thị dữ liệu lịch sử (màu cam) + forecast (màu đen)."""
    fig = go.Figure()
    # Dữ liệu lịch sử (màu cam)
    fig.add_trace(go.Scatter(
        x=history_series.index,
        y=history_series.values,
        mode='lines',
        name="Historical",
        line=dict(color="orange")
    ))
    # Forecast (màu đen, nét đứt)
    fig.add_trace(go.Scatter(
        x=forecast_df["Date"],
        y=forecast_df["Forecast"],
        mode='lines',
        name="Forecast",
        line=dict(dash="dash", color="black")
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="Quantity",
        hovermode="x unified",
        margin=dict(l=40, r=20, t=60, b=40)
    )
    return fig

#########################################
# 3. PIPELINE CHO GROUP X (ĐƠN GIẢN)
#########################################
def pipeline_x_forecast(series, months=6):
    """Dự báo future cho 1 series nhóm X."""
    if len(series) < 3:
        return None

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # Ensemble (50-50)
    ensemble_pred = 0.5 * forecast_arima + 0.5 * forecast_hw

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    df_forecast = pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})
    return df_forecast

def forecast_future_x(col, months):
    """Xử lý logic forecast cho 1 cột hoặc 'All' (cộng dồn) của Group X."""
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_x_cols:
            s = df[c].dropna()
            fc = pipeline_x_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        fc = pipeline_x_forecast(s, months)
        return fc

def plot_forecast_x(col, months):
    forecast_df = forecast_future_x(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()

    if col == "All":
        # Lịch sử cộng dồn
        hist = df[group_x_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group X - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group X - {col}")
        return fig

def gradio_predict_x(col, months):
    months = int(months)
    fc_df = forecast_future_x(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_x(col, months)
    return fc_df, fig

#########################################
# 4. PIPELINE CHO GROUP Y (TƯƠNG TỰ)
#########################################
def pipeline_y_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # XGB
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_xgb = XGBRegressor(n_estimators=50, learning_rate=0.1)
        model_xgb.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_xgb = model_xgb.predict(future_X)
    except:
        forecast_xgb = np.full(months, np.nan)

    # Ensemble (trung bình 3)
    arr_hw = np.nan_to_num(forecast_hw)
    arr_arima = np.nan_to_num(forecast_arima)
    arr_xgb = np.nan_to_num(forecast_xgb)
    ensemble_pred = (arr_hw + arr_arima + arr_xgb) / 3

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_y(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_y_cols:
            s = df[c].dropna()
            fc = pipeline_y_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_y_forecast(s, months)

def plot_forecast_y(col, months):
    forecast_df = forecast_future_y(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_y_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Y - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Y - {col}")
        return fig

def gradio_predict_y(col, months):
    months = int(months)
    fc_df = forecast_future_y(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_y(col, months)
    return fc_df, fig

#########################################
# 5. PIPELINE CHO GROUP Z (TƯƠNG TỰ)
#########################################
def pipeline_z_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Model 1: SARIMAX
    try:
        model_sarimax = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_sarimax = model_sarimax.predict(n_periods=months)
    except:
        forecast_sarimax = np.full(months, np.nan)

    # Model 2: GradientBoost
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05)
        model_gbr.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_gbr = model_gbr.predict(future_X)
    except:
        forecast_gbr = np.full(months, np.nan)

    # Model 3: Ridge
    try:
        model_ridge = Ridge(alpha=1.0, random_state=42)
        model_ridge.fit(X, y)
        future_ridge = model_ridge.predict(future_X)
    except:
        future_ridge = np.full(months, np.nan)

    # Model 4: Naive
    naive_val = series.iloc[-1] if len(series) > 0 else 0
    forecast_naive = np.full(months, naive_val)

    # Ensemble median
    forecasts_stack = np.vstack([
        np.nan_to_num(forecast_sarimax),
        np.nan_to_num(forecast_gbr),
        np.nan_to_num(future_ridge),
        np.nan_to_num(forecast_naive)
    ])
    ensemble_pred = np.median(forecasts_stack, axis=0)

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_z(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_z_cols:
            s = df[c].dropna()
            fc = pipeline_z_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_z_forecast(s, months)

def plot_forecast_z(col, months):
    forecast_df = forecast_future_z(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_z_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Z - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Z - {col}")
        return fig

def gradio_predict_z(col, months):
    months = int(months)
    fc_df = forecast_future_z(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_z(col, months)
    return fc_df, fig

#########################################
# 6. GRADIO DEPLOYMENT (3 Tabs: X, Y, Z)
#########################################

# CSS: Chỉ ép màu chữ là đen (và nền trắng) cho các phần tử text,
# KHÔNG động đến stroke/fill của các phần tử svg (Plotly lines).
custom_css = """
/* Ép text đen, nền trắng cho các element văn bản, input, button... */
body, .gradio-container, .gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container button,
.gradio-container label,
.gradio-container p,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container h5,
.gradio-container h6,
.gradio-container div,
.gradio-container span,
.gradio-container strong,
.gradio-container em {
  background-color: #FFFFFF !important;
  color: #000000 !important;
}

/* Placeholder input cũng màu đen */
::placeholder {
  color: #000000 !important;
}

/* Chiều cao cho plot */
.gradio-plot {
  height: 500px !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Multi-Group Series Forecasting Deployment for Outdoor Products")

    with gr.Tabs():
        # --- TAB CHO GROUP X ---
        with gr.Tab("Group X"):
            gr.Markdown("**Forecast for columns ending with `_x`**")

            dropdown_x = gr.Dropdown(label="Select Column", choices=group_x_options, value="All")
            forecast_periods_x = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_x = gr.Button("Forecast")

            with gr.Row():
                out_table_x = gr.Dataframe(label="Forecast Table")
                out_plot_x = gr.Plot(label="Forecast Chart")

            btn_x.click(
                fn=gradio_predict_x,
                inputs=[dropdown_x, forecast_periods_x],
                outputs=[out_table_x, out_plot_x]
            )

        # --- TAB CHO GROUP Y ---
        with gr.Tab("Group Y"):
            gr.Markdown("**Forecast for columns ending with `_y`**")

            dropdown_y = gr.Dropdown(label="Select Column", choices=group_y_options, value="All")
            forecast_periods_y = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_y = gr.Button("Forecast")

            with gr.Row():
                out_table_y = gr.Dataframe(label="Forecast Table")
                out_plot_y = gr.Plot(label="Forecast Chart")

            btn_y.click(
                fn=gradio_predict_y,
                inputs=[dropdown_y, forecast_periods_y],
                outputs=[out_table_y, out_plot_y]
            )

        # --- TAB CHO GROUP Z ---
        with gr.Tab("Group Z"):
            gr.Markdown("**Forecast for columns ending with `_z`**")

            dropdown_z = gr.Dropdown(label="Select Column", choices=group_z_options, value="All")
            forecast_periods_z = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_z = gr.Button("Forecast")

            with gr.Row():
                out_table_z = gr.Dataframe(label="Forecast Table")
                out_plot_z = gr.Plot(label="Forecast Chart")

            btn_z.click(
                fn=gradio_predict_z,
                inputs=[dropdown_z, forecast_periods_z],
                outputs=[out_table_z, out_plot_z]
            )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1ac2a4c058b417b99a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 2. Triển khai model dự báo cho warehouses với gradio

In [ ]:
outdoor_data=pd.read_excel("Outdoor.xlsx")

In [ ]:
import pandas as pd

# -------------------------------
# Step 0: Data Preparation and Cleaning
# -------------------------------
df = outdoor_data[[
    "FiscalMonthLastDate", "Warehouse", "Series No", "Item SKU", "Order Quantity"
]].copy().dropna()

# Đổi tên cột: thay khoảng trắng bằng dấu "-" để dễ xử lý
df.columns = [col.replace(" ", "-") for col in df.columns]

# Chuyển FiscalMonthLastDate sang kiểu datetime
df["FiscalMonthLastDate"] = pd.to_datetime(df["FiscalMonthLastDate"])

# -------------------------------
# Step 1: Aggregate Data at the Series Level
# -------------------------------
# Chúng ta chỉ quan tâm đến cột "Series No". Mỗi hàng sẽ được gộp theo FiscalMonthLastDate và Series-No.
df_agg = df.groupby(["FiscalMonthLastDate", "Warehouse"])["Order-Quantity"].sum().reset_index()

# -------------------------------
# Step 2: Convert to Wide Format for Forecasting
# -------------------------------
# Pivot table: index = FiscalMonthLastDate, các cột là các giá trị của "Series-No"
df_wide = df_agg.pivot_table(
    index="FiscalMonthLastDate",
    columns="Warehouse",
    values="Order-Quantity",
    fill_value=0
)

# -------------------------------
# Step 3: Compute Hierarchical Aggregations (2-level: Total and Series)
# -------------------------------
# 3.1: Total (Overall Sum): tổng của tất cả các Series
df_total = df_wide.sum(axis=1).to_frame(name=("Total",))

# 3.2: Series-level: giữ lại dữ liệu của các Series, đặt tên cột thành MultiIndex (1 cấp)
df_series = df_wide.copy()
df_series.columns = pd.MultiIndex.from_tuples([(col,) for col in df_series.columns])

# -------------------------------
# Step 4: Merge Total and Series-level Data
# -------------------------------
df_hierarchical = pd.concat([df_total, df_series], axis=1)

# -------------------------------
# Step 5: Sort Columns Hierarchically
# -------------------------------
# Sắp xếp sao cho cột "Total" đứng đầu, sau đó các Series được sắp xếp theo tên (alphabet)
ordered_columns = sorted(
    df_hierarchical.columns,
    key=lambda x: (0 if x[0] == "Total" else 1, x[0])
)
df_hierarchical = df_hierarchical[ordered_columns]

# -------------------------------
# Step 6: Flatten MultiIndex Columns
# -------------------------------
# Vì các cột Series đã có MultiIndex 1 cấp, ta chỉ lấy phần tử đầu tiên của mỗi tuple
df_hierarchical.columns = [col[0] for col in df_hierarchical.columns]

# -------------------------------
# (Optional) Replace specific dates in the index if needed
# -------------------------------
df_hierarchical.index = df_hierarchical.index.to_series().replace({
    pd.Timestamp("2023-04-01"): pd.Timestamp("2023-03-25"),
    pd.Timestamp("2023-07-01"): pd.Timestamp("2023-06-24")
})

# -------------------------------
# Phần 2: Phân loại các Series thành nhóm a, b, c theo doanh số
# -------------------------------
series_cols = [col for col in df_hierarchical.columns if col != "Total"]

# Tính tổng doanh số cho mỗi Series
series_totals = df_hierarchical[series_cols].sum(axis=0).reset_index()
series_totals.columns = ['Series-No', 'Total_Sales']
overall_total = series_totals['Total_Sales'].sum()
n_total = len(series_totals)

# Sắp xếp giảm dần theo Total_Sales
series_totals = series_totals.sort_values(by='Total_Sales', ascending=False).reset_index(drop=True)

# Nhóm a: chọn các series theo thứ tự giảm dần cho đến khi tích lũy ≥80% overall sales,
#         nhưng số lượng không vượt quá 20% tổng số series.
max_count_a = int(0.2 * n_total)
cumulative_sales_a = 0
group_a = []
for idx, row in series_totals.iterrows():
    group_a.append(row['Series-No'])
    cumulative_sales_a += row['Total_Sales']
    if (cumulative_sales_a / overall_total >= 0.80) or (len(group_a) >= max_count_a):
        break

# Nhóm b: từ các series còn lại, chọn đến khi tích lũy ≥15% overall sales,
#         nhưng số lượng không vượt quá 30% số series còn lại.
remaining_df = series_totals[~series_totals['Series-No'].isin(group_a)].copy()
n_remaining = len(remaining_df)
max_count_b = int(0.3 * n_remaining)
cumulative_sales_b = 0
group_b = []
for idx, row in remaining_df.iterrows():
    group_b.append(row['Series-No'])
    cumulative_sales_b += row['Total_Sales']
    if (cumulative_sales_b / overall_total >= 0.15) or (len(group_b) >= max_count_b):
        break

# Nhóm c: các series còn lại
group_c = remaining_df[~remaining_df['Series-No'].isin(group_b)]['Series-No'].tolist()

series_totals['Group'] = series_totals['Series-No'].apply(
    lambda x: 'a' if x in group_a else ('b' if x in group_b else 'c')
)

print("\nPhân nhóm doanh số (a, b, c):")
print("Tổng số Series:", n_total)
print("Nhóm a: {} Series, chiếm {:.2%} doanh số".format(len(group_a), cumulative_sales_a / overall_total))
print("Nhóm b: {} Series, chiếm {:.2%} doanh số".format(len(group_b), cumulative_sales_b / overall_total))
remaining_sales = overall_total - cumulative_sales_a - cumulative_sales_b
print("Nhóm c: {} Series, chiếm {:.2%} doanh số".format(len(group_c), remaining_sales / overall_total))






# -------------------------------
# Phần 3: Phân nhóm bổ sung x, y, z trong mỗi nhóm a, b, c theo hệ số biến động (CV)
# -------------------------------
# Tính CV từ dữ liệu 12 tháng cuối của wide format
last_12 = df_hierarchical.iloc[-12:]
def compute_cv(series_data):
    mean_val = series_data.mean()
    return np.nan if mean_val == 0 else series_data.std() / mean_val

# Tính CV cho tất cả các series
cv_dict = {series: compute_cv(last_12[series]) for series in series_cols}

# Áp dụng ngưỡng phân loại CV: X: CV < 0.34, Y: 0.34 ≤ CV < 0.84, Z: CV ≥ 0.84.
subgroup_dict = {}
for series, cv_val in cv_dict.items():
    if pd.isna(cv_val):
        subgroup = 'unknown'
    elif cv_val < 0.34:
        subgroup = 'x'
    elif cv_val < 0.84:
        subgroup = 'y'
    else:
        subgroup = 'z'
    subgroup_dict[series] = subgroup

# In bảng tóm tắt số lượng các nhóm phụ
from collections import Counter
subgroup_counts = Counter(subgroup_dict.values())
print("\nPhân loại phụ theo CV:")
for key in ['x', 'y', 'z', 'unknown']:
    print(f"Nhóm {key}: {subgroup_counts.get(key, 0)} series")

# -------------------------------
# Phần 4: Gán nhãn kết hợp vào tên cột wide format
# -------------------------------
# Tạo mapping cuối: tên cột sẽ có định dạng <Series-No>_<Group (a,b,c)>_<Subgroup (x,y,z,unknown)>
final_mapping = {}
for series in series_cols:
    # Lấy nhãn nhóm a,b,c từ series_totals
    group_label = series_totals.loc[series_totals['Series-No'] == series, 'Group'].values[0]
    subgroup_label = subgroup_dict.get(series, '')
    final_mapping[series] = f"{series}_{group_label}_{subgroup_label}"

# Cập nhật tên cột trong df_hierarchical (cột Total giữ nguyên)
new_columns = {}
for col in df_hierarchical.columns:
    if col == "Total":
        new_columns[col] = col
    else:
        new_columns[col] = final_mapping.get(col, col)

df_final = df_hierarchical.rename(columns=new_columns)


# -------------------------------
# Phần 5: Tách Stop Produced Series và Valid Series
# -------------------------------
# Lấy dữ liệu 12 tháng cuối
last_12 = df_final.iloc[-12:]

# Xác định các cột mà toàn bộ 12 giá trị cuối cùng đều bằng 0
stop_columns = [col for col in df_final.columns if (last_12[col] == 0).all()]

# Tạo DataFrame chứa các stop produced series và valid series
df_zero = df_final[stop_columns]
df_valid = df_final.drop(columns=stop_columns)

# In ra số lượng các series từng loại
print("Số lượng stop produced series:", len(stop_columns))
print("Số lượng valid series:", len(df_valid.columns) - (1 if 'Total' in df_valid.columns else 0)-1)

# -------------------------------
# Lưu kết quả thành 2 file CSV
# -------------------------------
df_zero.to_csv("Stopped_Warehouse_Hierarchical.csv", index=True)
df_valid.to_csv("Warehouse_Hierarchical.csv", index=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# Modeling
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Deep learning
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Gradio
import gradio as gr

#########################################
# 1. ĐỌC & CHUẨN BỊ DỮ LIỆU
#########################################
df = pd.read_csv('Warehouse_Hierarchical.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)

group_x_cols = [col for col in df.columns if col.endswith('_x')]
group_y_cols = [col for col in df.columns if col.endswith('_y')]
group_z_cols = [col for col in df.columns if col.endswith('_z')]

# Thêm "All" vào dropdown
group_x_options = ["All"] + group_x_cols
group_y_options = ["All"] + group_y_cols
group_z_options = ["All"] + group_z_cols

#########################################
# 2. HÀM TIỆN ÍCH CHUNG
#########################################
def build_future_dates(last_date, periods):
    """Tạo dãy datetime hàng tháng, bắt đầu từ tháng tiếp theo của last_date."""
    return pd.date_range(start=last_date + pd.DateOffset(months=1),
                         periods=periods, freq="M")

def plot_history_and_forecast(history_series, forecast_df, title):
    """Trả về plotly figure hiển thị dữ liệu lịch sử (màu cam) + forecast (màu đen)."""
    fig = go.Figure()
    # Dữ liệu lịch sử (màu cam)
    fig.add_trace(go.Scatter(
        x=history_series.index,
        y=history_series.values,
        mode='lines',
        name="Historical",
        line=dict(color="orange")
    ))
    # Forecast (màu đen, nét đứt)
    fig.add_trace(go.Scatter(
        x=forecast_df["Date"],
        y=forecast_df["Forecast"],
        mode='lines',
        name="Forecast",
        line=dict(dash="dash", color="black")
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="Quantity",
        hovermode="x unified",
        margin=dict(l=40, r=20, t=60, b=40)
    )
    return fig

#########################################
# 3. PIPELINE CHO GROUP X (ĐƠN GIẢN)
#########################################
def pipeline_x_forecast(series, months=6):
    """Dự báo future cho 1 series nhóm X."""
    if len(series) < 3:
        return None

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # Ensemble (50-50)
    ensemble_pred = 0.5 * forecast_arima + 0.5 * forecast_hw

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    df_forecast = pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})
    return df_forecast

def forecast_future_x(col, months):
    """Xử lý logic forecast cho 1 cột hoặc 'All' (cộng dồn) của Group X."""
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_x_cols:
            s = df[c].dropna()
            fc = pipeline_x_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        fc = pipeline_x_forecast(s, months)
        return fc

def plot_forecast_x(col, months):
    forecast_df = forecast_future_x(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()

    if col == "All":
        # Lịch sử cộng dồn
        hist = df[group_x_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group X - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group X - {col}")
        return fig

def gradio_predict_x(col, months):
    months = int(months)
    fc_df = forecast_future_x(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_x(col, months)
    return fc_df, fig

#########################################
# 4. PIPELINE CHO GROUP Y (TƯƠNG TỰ)
#########################################
def pipeline_y_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # XGB
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_xgb = XGBRegressor(n_estimators=50, learning_rate=0.1)
        model_xgb.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_xgb = model_xgb.predict(future_X)
    except:
        forecast_xgb = np.full(months, np.nan)

    # Ensemble (trung bình 3)
    arr_hw = np.nan_to_num(forecast_hw)
    arr_arima = np.nan_to_num(forecast_arima)
    arr_xgb = np.nan_to_num(forecast_xgb)
    ensemble_pred = (arr_hw + arr_arima + arr_xgb) / 3

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_y(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_y_cols:
            s = df[c].dropna()
            fc = pipeline_y_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_y_forecast(s, months)

def plot_forecast_y(col, months):
    forecast_df = forecast_future_y(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_y_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Y - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Y - {col}")
        return fig

def gradio_predict_y(col, months):
    months = int(months)
    fc_df = forecast_future_y(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_y(col, months)
    return fc_df, fig

#########################################
# 5. PIPELINE CHO GROUP Z (TƯƠNG TỰ)
#########################################
def pipeline_z_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Model 1: SARIMAX
    try:
        model_sarimax = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_sarimax = model_sarimax.predict(n_periods=months)
    except:
        forecast_sarimax = np.full(months, np.nan)

    # Model 2: GradientBoost
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05)
        model_gbr.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_gbr = model_gbr.predict(future_X)
    except:
        forecast_gbr = np.full(months, np.nan)

    # Model 3: Ridge
    try:
        model_ridge = Ridge(alpha=1.0, random_state=42)
        model_ridge.fit(X, y)
        future_ridge = model_ridge.predict(future_X)
    except:
        future_ridge = np.full(months, np.nan)

    # Model 4: Naive
    naive_val = series.iloc[-1] if len(series) > 0 else 0
    forecast_naive = np.full(months, naive_val)

    # Ensemble median
    forecasts_stack = np.vstack([
        np.nan_to_num(forecast_sarimax),
        np.nan_to_num(forecast_gbr),
        np.nan_to_num(future_ridge),
        np.nan_to_num(forecast_naive)
    ])
    ensemble_pred = np.median(forecasts_stack, axis=0)

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_z(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_z_cols:
            s = df[c].dropna()
            fc = pipeline_z_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_z_forecast(s, months)

def plot_forecast_z(col, months):
    forecast_df = forecast_future_z(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_z_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Z - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Z - {col}")
        return fig

def gradio_predict_z(col, months):
    months = int(months)
    fc_df = forecast_future_z(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_z(col, months)
    return fc_df, fig

#########################################
# 6. GRADIO DEPLOYMENT (3 Tabs: X, Y, Z)
#########################################

# CSS: Chỉ ép màu chữ là đen (và nền trắng) cho các phần tử text,
# KHÔNG động đến stroke/fill của các phần tử svg (Plotly lines).
custom_css = """
/* Ép text đen, nền trắng cho các element văn bản, input, button... */
body, .gradio-container, .gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container button,
.gradio-container label,
.gradio-container p,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container h5,
.gradio-container h6,
.gradio-container div,
.gradio-container span,
.gradio-container strong,
.gradio-container em {
  background-color: #FFFFFF !important;
  color: #000000 !important;
}

/* Placeholder input cũng màu đen */
::placeholder {
  color: #000000 !important;
}

/* Chiều cao cho plot */
.gradio-plot {
  height: 500px !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Multi-Group Warehouse Forecasting Deployment for Outdoor Products")

    with gr.Tabs():
        # --- TAB CHO GROUP X ---
        with gr.Tab("Group X"):
            gr.Markdown("**Forecast for columns ending with `_x`**")

            dropdown_x = gr.Dropdown(label="Select Column", choices=group_x_options, value="All")
            forecast_periods_x = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_x = gr.Button("Forecast")

            with gr.Row():
                out_table_x = gr.Dataframe(label="Forecast Table")
                out_plot_x = gr.Plot(label="Forecast Chart")

            btn_x.click(
                fn=gradio_predict_x,
                inputs=[dropdown_x, forecast_periods_x],
                outputs=[out_table_x, out_plot_x]
            )

        # --- TAB CHO GROUP Y ---
        with gr.Tab("Group Y"):
            gr.Markdown("**Forecast for columns ending with `_y`**")

            dropdown_y = gr.Dropdown(label="Select Column", choices=group_y_options, value="All")
            forecast_periods_y = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_y = gr.Button("Forecast")

            with gr.Row():
                out_table_y = gr.Dataframe(label="Forecast Table")
                out_plot_y = gr.Plot(label="Forecast Chart")

            btn_y.click(
                fn=gradio_predict_y,
                inputs=[dropdown_y, forecast_periods_y],
                outputs=[out_table_y, out_plot_y]
            )

        # --- TAB CHO GROUP Z ---
        with gr.Tab("Group Z"):
            gr.Markdown("**Forecast for columns ending with `_z`**")

            dropdown_z = gr.Dropdown(label="Select Column", choices=group_z_options, value="All")
            forecast_periods_z = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_z = gr.Button("Forecast")

            with gr.Row():
                out_table_z = gr.Dataframe(label="Forecast Table")
                out_plot_z = gr.Plot(label="Forecast Chart")

            btn_z.click(
                fn=gradio_predict_z,
                inputs=[dropdown_z, forecast_periods_z],
                outputs=[out_table_z, out_plot_z]
            )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7e46e7e58fcad56a0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 3. Triển khai model dự báo cho SKUs với gradio

In [ ]:
outdoor_data=pd.read_excel("Outdoor.xlsx")

In [ ]:
import pandas as pd

# -------------------------------
# Step 0: Data Preparation and Cleaning
# -------------------------------
df = outdoor_data[[
    "FiscalMonthLastDate", "Warehouse", "Series No", "Item SKU", "Order Quantity"
]].copy().dropna()

# Đổi tên cột: thay khoảng trắng bằng dấu "-" để dễ xử lý
df.columns = [col.replace(" ", "-") for col in df.columns]

# Chuyển FiscalMonthLastDate sang kiểu datetime
df["FiscalMonthLastDate"] = pd.to_datetime(df["FiscalMonthLastDate"])

# -------------------------------
# Step 1: Aggregate Data at the Series Level
# -------------------------------
# Chúng ta chỉ quan tâm đến cột "Series No". Mỗi hàng sẽ được gộp theo FiscalMonthLastDate và Series-No.
df_agg = df.groupby(["FiscalMonthLastDate", "Item-SKU"])["Order-Quantity"].sum().reset_index()

# -------------------------------
# Step 2: Convert to Wide Format for Forecasting
# -------------------------------
# Pivot table: index = FiscalMonthLastDate, các cột là các giá trị của "Series-No"
df_wide = df_agg.pivot_table(
    index="FiscalMonthLastDate",
    columns="Item-SKU",
    values="Order-Quantity",
    fill_value=0
)

# -------------------------------
# Step 3: Compute Hierarchical Aggregations (2-level: Total and Series)
# -------------------------------
# 3.1: Total (Overall Sum): tổng của tất cả các Series
df_total = df_wide.sum(axis=1).to_frame(name=("Total",))

# 3.2: Series-level: giữ lại dữ liệu của các Series, đặt tên cột thành MultiIndex (1 cấp)
df_series = df_wide.copy()
df_series.columns = pd.MultiIndex.from_tuples([(col,) for col in df_series.columns])

# -------------------------------
# Step 4: Merge Total and Series-level Data
# -------------------------------
df_hierarchical = pd.concat([df_total, df_series], axis=1)

# -------------------------------
# Step 5: Sort Columns Hierarchically
# -------------------------------
# Sắp xếp sao cho cột "Total" đứng đầu, sau đó các Series được sắp xếp theo tên (alphabet)
ordered_columns = sorted(
    df_hierarchical.columns,
    key=lambda x: (0 if x[0] == "Total" else 1, x[0])
)
df_hierarchical = df_hierarchical[ordered_columns]

# -------------------------------
# Step 6: Flatten MultiIndex Columns
# -------------------------------
# Vì các cột Series đã có MultiIndex 1 cấp, ta chỉ lấy phần tử đầu tiên của mỗi tuple
df_hierarchical.columns = [col[0] for col in df_hierarchical.columns]

# -------------------------------
# (Optional) Replace specific dates in the index if needed
# -------------------------------
df_hierarchical.index = df_hierarchical.index.to_series().replace({
    pd.Timestamp("2023-04-01"): pd.Timestamp("2023-03-25"),
    pd.Timestamp("2023-07-01"): pd.Timestamp("2023-06-24")
})

# -------------------------------
# Phần 2: Phân loại các Series thành nhóm a, b, c theo doanh số
# -------------------------------
series_cols = [col for col in df_hierarchical.columns if col != "Total"]

# Tính tổng doanh số cho mỗi Series
series_totals = df_hierarchical[series_cols].sum(axis=0).reset_index()
series_totals.columns = ['Series-No', 'Total_Sales']
overall_total = series_totals['Total_Sales'].sum()
n_total = len(series_totals)

# Sắp xếp giảm dần theo Total_Sales
series_totals = series_totals.sort_values(by='Total_Sales', ascending=False).reset_index(drop=True)

# Nhóm a: chọn các series theo thứ tự giảm dần cho đến khi tích lũy ≥80% overall sales,
#         nhưng số lượng không vượt quá 20% tổng số series.
max_count_a = int(0.2 * n_total)
cumulative_sales_a = 0
group_a = []
for idx, row in series_totals.iterrows():
    group_a.append(row['Series-No'])
    cumulative_sales_a += row['Total_Sales']
    if (cumulative_sales_a / overall_total >= 0.80) or (len(group_a) >= max_count_a):
        break

# Nhóm b: từ các series còn lại, chọn đến khi tích lũy ≥15% overall sales,
#         nhưng số lượng không vượt quá 30% số series còn lại.
remaining_df = series_totals[~series_totals['Series-No'].isin(group_a)].copy()
n_remaining = len(remaining_df)
max_count_b = int(0.3 * n_remaining)
cumulative_sales_b = 0
group_b = []
for idx, row in remaining_df.iterrows():
    group_b.append(row['Series-No'])
    cumulative_sales_b += row['Total_Sales']
    if (cumulative_sales_b / overall_total >= 0.15) or (len(group_b) >= max_count_b):
        break

# Nhóm c: các series còn lại
group_c = remaining_df[~remaining_df['Series-No'].isin(group_b)]['Series-No'].tolist()

series_totals['Group'] = series_totals['Series-No'].apply(
    lambda x: 'a' if x in group_a else ('b' if x in group_b else 'c')
)

print("\nPhân nhóm doanh số (a, b, c):")
print("Tổng số Series:", n_total)
print("Nhóm a: {} Series, chiếm {:.2%} doanh số".format(len(group_a), cumulative_sales_a / overall_total))
print("Nhóm b: {} Series, chiếm {:.2%} doanh số".format(len(group_b), cumulative_sales_b / overall_total))
remaining_sales = overall_total - cumulative_sales_a - cumulative_sales_b
print("Nhóm c: {} Series, chiếm {:.2%} doanh số".format(len(group_c), remaining_sales / overall_total))






# -------------------------------
# Phần 3: Phân nhóm bổ sung x, y, z trong mỗi nhóm a, b, c theo hệ số biến động (CV)
# -------------------------------
# Tính CV từ dữ liệu 12 tháng cuối của wide format
last_12 = df_hierarchical.iloc[-12:]
def compute_cv(series_data):
    mean_val = series_data.mean()
    return np.nan if mean_val == 0 else series_data.std() / mean_val

# Tính CV cho tất cả các series
cv_dict = {series: compute_cv(last_12[series]) for series in series_cols}

# Áp dụng ngưỡng phân loại CV: X: CV < 0.34, Y: 0.34 ≤ CV < 0.84, Z: CV ≥ 0.84.
subgroup_dict = {}
for series, cv_val in cv_dict.items():
    if pd.isna(cv_val):
        subgroup = 'unknown'
    elif cv_val < 0.34:
        subgroup = 'x'
    elif cv_val < 0.84:
        subgroup = 'y'
    else:
        subgroup = 'z'
    subgroup_dict[series] = subgroup

# In bảng tóm tắt số lượng các nhóm phụ
from collections import Counter
subgroup_counts = Counter(subgroup_dict.values())
print("\nPhân loại phụ theo CV:")
for key in ['x', 'y', 'z', 'unknown']:
    print(f"Nhóm {key}: {subgroup_counts.get(key, 0)} series")

# -------------------------------
# Phần 4: Gán nhãn kết hợp vào tên cột wide format
# -------------------------------
# Tạo mapping cuối: tên cột sẽ có định dạng <Series-No>_<Group (a,b,c)>_<Subgroup (x,y,z,unknown)>
final_mapping = {}
for series in series_cols:
    # Lấy nhãn nhóm a,b,c từ series_totals
    group_label = series_totals.loc[series_totals['Series-No'] == series, 'Group'].values[0]
    subgroup_label = subgroup_dict.get(series, '')
    final_mapping[series] = f"{series}_{group_label}_{subgroup_label}"

# Cập nhật tên cột trong df_hierarchical (cột Total giữ nguyên)
new_columns = {}
for col in df_hierarchical.columns:
    if col == "Total":
        new_columns[col] = col
    else:
        new_columns[col] = final_mapping.get(col, col)

df_final = df_hierarchical.rename(columns=new_columns)


# -------------------------------
# Phần 5: Tách Stop Produced Series và Valid Series
# -------------------------------
# Lấy dữ liệu 12 tháng cuối
last_12 = df_final.iloc[-12:]

# Xác định các cột mà toàn bộ 12 giá trị cuối cùng đều bằng 0
stop_columns = [col for col in df_final.columns if (last_12[col] == 0).all()]

# Tạo DataFrame chứa các stop produced series và valid series
df_zero = df_final[stop_columns]
df_valid = df_final.drop(columns=stop_columns)

# -------------------------------
# Lưu kết quả thành 2 file CSV
# -------------------------------
df_zero.to_csv("Stopped_Sku_Hierarchical.csv", index=True)
df_valid.to_csv("Sku_Hierarchical.csv", index=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# Modeling
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Deep learning
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Gradio
import gradio as gr

#########################################
# 1. ĐỌC & CHUẨN BỊ DỮ LIỆU
#########################################
df = pd.read_csv('Sku_Hierarchical.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)

group_x_cols = [col for col in df.columns if col.endswith('_x')]
group_y_cols = [col for col in df.columns if col.endswith('_y')]
group_z_cols = [col for col in df.columns if col.endswith('_z')]

# Thêm "All" vào dropdown
group_x_options = ["All"] + group_x_cols
group_y_options = ["All"] + group_y_cols
group_z_options = ["All"] + group_z_cols

#########################################
# 2. HÀM TIỆN ÍCH CHUNG
#########################################
def build_future_dates(last_date, periods):
    """Tạo dãy datetime hàng tháng, bắt đầu từ tháng tiếp theo của last_date."""
    return pd.date_range(start=last_date + pd.DateOffset(months=1),
                         periods=periods, freq="M")

def plot_history_and_forecast(history_series, forecast_df, title):
    """Trả về plotly figure hiển thị dữ liệu lịch sử (màu cam) + forecast (màu đen)."""
    fig = go.Figure()
    # Dữ liệu lịch sử (màu cam)
    fig.add_trace(go.Scatter(
        x=history_series.index,
        y=history_series.values,
        mode='lines',
        name="Historical",
        line=dict(color="orange")
    ))
    # Forecast (màu đen, nét đứt)
    fig.add_trace(go.Scatter(
        x=forecast_df["Date"],
        y=forecast_df["Forecast"],
        mode='lines',
        name="Forecast",
        line=dict(dash="dash", color="black")
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="Quantity",
        hovermode="x unified",
        margin=dict(l=40, r=20, t=60, b=40)
    )
    return fig

#########################################
# 3. PIPELINE CHO GROUP X (ĐƠN GIẢN)
#########################################
def pipeline_x_forecast(series, months=6):
    """Dự báo future cho 1 series nhóm X."""
    if len(series) < 3:
        return None

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # Ensemble (50-50)
    ensemble_pred = 0.5 * forecast_arima + 0.5 * forecast_hw

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    df_forecast = pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})
    return df_forecast

def forecast_future_x(col, months):
    """Xử lý logic forecast cho 1 cột hoặc 'All' (cộng dồn) của Group X."""
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_x_cols:
            s = df[c].dropna()
            fc = pipeline_x_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        fc = pipeline_x_forecast(s, months)
        return fc

def plot_forecast_x(col, months):
    forecast_df = forecast_future_x(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()

    if col == "All":
        # Lịch sử cộng dồn
        hist = df[group_x_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group X - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group X - {col}")
        return fig

def gradio_predict_x(col, months):
    months = int(months)
    fc_df = forecast_future_x(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_x(col, months)
    return fc_df, fig

#########################################
# 4. PIPELINE CHO GROUP Y (TƯƠNG TỰ)
#########################################
def pipeline_y_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # XGB
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_xgb = XGBRegressor(n_estimators=50, learning_rate=0.1)
        model_xgb.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_xgb = model_xgb.predict(future_X)
    except:
        forecast_xgb = np.full(months, np.nan)

    # Ensemble (trung bình 3)
    arr_hw = np.nan_to_num(forecast_hw)
    arr_arima = np.nan_to_num(forecast_arima)
    arr_xgb = np.nan_to_num(forecast_xgb)
    ensemble_pred = (arr_hw + arr_arima + arr_xgb) / 3

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_y(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_y_cols:
            s = df[c].dropna()
            fc = pipeline_y_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_y_forecast(s, months)

def plot_forecast_y(col, months):
    forecast_df = forecast_future_y(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_y_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Y - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Y - {col}")
        return fig

def gradio_predict_y(col, months):
    months = int(months)
    fc_df = forecast_future_y(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_y(col, months)
    return fc_df, fig

#########################################
# 5. PIPELINE CHO GROUP Z (TƯƠNG TỰ)
#########################################
def pipeline_z_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Model 1: SARIMAX
    try:
        model_sarimax = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_sarimax = model_sarimax.predict(n_periods=months)
    except:
        forecast_sarimax = np.full(months, np.nan)

    # Model 2: GradientBoost
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05)
        model_gbr.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_gbr = model_gbr.predict(future_X)
    except:
        forecast_gbr = np.full(months, np.nan)

    # Model 3: Ridge
    try:
        model_ridge = Ridge(alpha=1.0, random_state=42)
        model_ridge.fit(X, y)
        future_ridge = model_ridge.predict(future_X)
    except:
        future_ridge = np.full(months, np.nan)

    # Model 4: Naive
    naive_val = series.iloc[-1] if len(series) > 0 else 0
    forecast_naive = np.full(months, naive_val)

    # Ensemble median
    forecasts_stack = np.vstack([
        np.nan_to_num(forecast_sarimax),
        np.nan_to_num(forecast_gbr),
        np.nan_to_num(future_ridge),
        np.nan_to_num(forecast_naive)
    ])
    ensemble_pred = np.median(forecasts_stack, axis=0)

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_z(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_z_cols:
            s = df[c].dropna()
            fc = pipeline_z_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_z_forecast(s, months)

def plot_forecast_z(col, months):
    forecast_df = forecast_future_z(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_z_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Z - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Z - {col}")
        return fig

def gradio_predict_z(col, months):
    months = int(months)
    fc_df = forecast_future_z(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_z(col, months)
    return fc_df, fig

#########################################
# 6. GRADIO DEPLOYMENT (3 Tabs: X, Y, Z)
#########################################

# CSS: Chỉ ép màu chữ là đen (và nền trắng) cho các phần tử text,
# KHÔNG động đến stroke/fill của các phần tử svg (Plotly lines).
custom_css = """
/* Ép text đen, nền trắng cho các element văn bản, input, button... */
body, .gradio-container, .gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container button,
.gradio-container label,
.gradio-container p,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container h5,
.gradio-container h6,
.gradio-container div,
.gradio-container span,
.gradio-container strong,
.gradio-container em {
  background-color: #FFFFFF !important;
  color: #000000 !important;
}

/* Placeholder input cũng màu đen */
::placeholder {
  color: #000000 !important;
}

/* Chiều cao cho plot */
.gradio-plot {
  height: 500px !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Multi-Group SKU Forecasting Deployment for Outdoor Products")

    with gr.Tabs():
        # --- TAB CHO GROUP X ---
        with gr.Tab("Group X"):
            gr.Markdown("**Forecast for columns ending with `_x`**")

            dropdown_x = gr.Dropdown(label="Select Column", choices=group_x_options, value="All")
            forecast_periods_x = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_x = gr.Button("Forecast")

            with gr.Row():
                out_table_x = gr.Dataframe(label="Forecast Table")
                out_plot_x = gr.Plot(label="Forecast Chart")

            btn_x.click(
                fn=gradio_predict_x,
                inputs=[dropdown_x, forecast_periods_x],
                outputs=[out_table_x, out_plot_x]
            )

        # --- TAB CHO GROUP Y ---
        with gr.Tab("Group Y"):
            gr.Markdown("**Forecast for columns ending with `_y`**")

            dropdown_y = gr.Dropdown(label="Select Column", choices=group_y_options, value="All")
            forecast_periods_y = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_y = gr.Button("Forecast")

            with gr.Row():
                out_table_y = gr.Dataframe(label="Forecast Table")
                out_plot_y = gr.Plot(label="Forecast Chart")

            btn_y.click(
                fn=gradio_predict_y,
                inputs=[dropdown_y, forecast_periods_y],
                outputs=[out_table_y, out_plot_y]
            )

        # --- TAB CHO GROUP Z ---
        with gr.Tab("Group Z"):
            gr.Markdown("**Forecast for columns ending with `_z`**")

            dropdown_z = gr.Dropdown(label="Select Column", choices=group_z_options, value="All")
            forecast_periods_z = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_z = gr.Button("Forecast")

            with gr.Row():
                out_table_z = gr.Dataframe(label="Forecast Table")
                out_plot_z = gr.Plot(label="Forecast Chart")

            btn_z.click(
                fn=gradio_predict_z,
                inputs=[dropdown_z, forecast_periods_z],
                outputs=[out_table_z, out_plot_z]
            )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e19d40e69a90038ce.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
